In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "NVDA", "TSLA"]

print("AVA-AI Version 2 — Advanced Features")
print("Raw data directory:", RAW_DIR.resolve())
print("Tickers:", ", ".join(tickers))

AVA-AI Version 2 — Advanced Features
Raw data directory: C:\Users\Aryan\Desktop\AVA-AI_MODEL-2\data\raw
Tickers: AAPL, AMZN, GOOGL, MSFT, NVDA, TSLA


In [2]:
stocks = {}

for ticker in tickers:
    file_path = RAW_DIR / f"{ticker}_daily.csv"
    df = pd.read_csv(file_path, parse_dates=["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    stocks[ticker] = df

print("Loaded stocks:")
for ticker, df in stocks.items():
    print(f"{ticker}: {len(df)} rows")

Loaded stocks:
AAPL: 500 rows
AMZN: 500 rows
GOOGL: 500 rows
MSFT: 500 rows
NVDA: 500 rows
TSLA: 500 rows


In [3]:
def add_v2_features(df):
    df = df.copy()

    # Basic returns
    df["return"] = df["close"].pct_change()
    df["return_lag1"] = df["return"].shift(1)
    df["return_lag2"] = df["return"].shift(2)
    df["return_lag3"] = df["return"].shift(3)
    df["return_lag5"] = df["return"].shift(5)
    df["return_lag10"] = df["return"].shift(10)

    # Moving averages
    df["MA5"] = df["close"].rolling(5).mean()
    df["MA20"] = df["close"].rolling(20).mean()
    df["MA50"] = df["close"].rolling(50).mean()

    # Price relative to moving averages
    df["price_vs_MA5"] = df["close"] / df["MA5"] - 1
    df["price_vs_MA20"] = df["close"] / df["MA20"] - 1
    df["price_vs_MA50"] = df["close"] / df["MA50"] - 1

    # Rolling returns
    df["rolling_return_5"] = df["return"].rolling(5).mean()
    df["rolling_return_10"] = df["return"].rolling(10).mean()
    df["rolling_return_20"] = df["return"].rolling(20).mean()

    # Volatility
    df["volatility_5"] = df["return"].rolling(5).std()
    df["volatility_10"] = df["return"].rolling(10).std()
    df["volatility_20"] = df["return"].rolling(20).std()

    # Volume
    df["volume_change"] = df["volume"].pct_change()
    df["volume_MA20"] = df["volume"].rolling(20).mean()
    df["volume_vs_MA20"] = df["volume"] / df["volume_MA20"] - 1

    # RSI
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_histogram"] = df["MACD"] - df["MACD_signal"]

    # Target: next-day return
    df["target_return"] = df["close"].shift(-1) / df["close"] - 1

    return df


for ticker in tickers:
    stocks[ticker] = add_v2_features(stocks[ticker])

print("V2 feature engineering complete.")
print("AAPL columns:", len(stocks["AAPL"].columns))
print(stocks["AAPL"].columns.tolist())

V2 feature engineering complete.
AAPL columns: 32
['datetime', 'open', 'high', 'low', 'close', 'volume', 'return', 'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5', 'return_lag10', 'MA5', 'MA20', 'MA50', 'price_vs_MA5', 'price_vs_MA20', 'price_vs_MA50', 'rolling_return_5', 'rolling_return_10', 'rolling_return_20', 'volatility_5', 'volatility_10', 'volatility_20', 'volume_change', 'volume_MA20', 'volume_vs_MA20', 'RSI_14', 'MACD', 'MACD_signal', 'MACD_histogram', 'target_return']


In [4]:
v2_features = [
    "close",
    "MA5",
    "MA20",
    "MA50",
    "price_vs_MA5",
    "price_vs_MA20",
    "price_vs_MA50",
    "volatility_5",
    "volatility_10",
    "volatility_20",
    "volume_change",
    "volume_vs_MA20",
    "return_lag1",
    "return_lag2",
    "return_lag3",
    "return_lag5",
    "return_lag10",
    "rolling_return_5",
    "rolling_return_10",
    "rolling_return_20",
    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_histogram",
]

print("V2 feature count:", len(v2_features))
print("Features:")
for feature in v2_features:
    print("-", feature)

V2 feature count: 24
Features:
- close
- MA5
- MA20
- MA50
- price_vs_MA5
- price_vs_MA20
- price_vs_MA50
- volatility_5
- volatility_10
- volatility_20
- volume_change
- volume_vs_MA20
- return_lag1
- return_lag2
- return_lag3
- return_lag5
- return_lag10
- rolling_return_5
- rolling_return_10
- rolling_return_20
- RSI_14
- MACD
- MACD_signal
- MACD_histogram


In [5]:
walk_splits = [
    (150, 150, 210),
    (210, 210, 270),
    (270, 270, 330),
    (330, 330, 390),
    (390, 390, 450),
]

prepared_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(subset=v2_features + ["target_return"]).copy()

    prepared_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_features)}"
    )

AAPL: 450 usable rows | features=24
AMZN: 450 usable rows | features=24
GOOGL: 450 usable rows | features=24
MSFT: 450 usable rows | features=24
NVDA: 450 usable rows | features=24
TSLA: 450 usable rows | features=24


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

v2_results = []

for ticker in tickers:
    df = prepared_data[ticker]

    X = df[v2_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(mean_absolute_error(y_test, predictions))
        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    v2_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

v2_results_df = pd.DataFrame(v2_results)

print(v2_results_df.to_string(index=False))

print("\nV2 overall:")
print("MAE :", v2_results_df["MAE"].mean())
print("RMSE:", v2_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.013948 0.018329
  AMZN 0.017560 0.023984
 GOOGL 0.019229 0.024829
  MSFT 0.014142 0.019471
  NVDA 0.019300 0.024581
  TSLA 0.025086 0.031960

V2 overall:
MAE : 0.01821080903151551
RMSE: 0.023858789032059863


## Experiment 1 — Expanded Technical Features

V2 expanded the technical feature set from the Base Model by adding:

- Shorter moving averages
- Additional return lags
- Multiple volatility windows
- Volume-relative features
- Rolling return features
- MACD histogram
- Price-to-moving-average relationships

The Random Forest model was evaluated using the same walk-forward validation framework as the Base Model.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.013948 | 0.018329 |
| AMZN | 0.017560 | 0.023984 |
| GOOGL | 0.019229 | 0.024829 |
| MSFT | 0.014142 | 0.019471 |
| NVDA | 0.019300 | 0.024581 |
| TSLA | 0.025086 | 0.031960 |

Overall:

- MAE: **0.018211**
- RMSE: **0.023859**

Compared with the V1 Random Forest, the expanded feature set did not improve the overall average error.

The results were mixed across individual stocks, with improvements for AAPL, AMZN, GOOGL, NVDA and TSLA, while MSFT was slightly worse.

### Interpretation

Adding more technical indicators alone did not provide consistent evidence of improved predictive performance.

This experiment will therefore be treated as an exploratory V2 result rather than a replacement for the Base Model.

The next experiment will introduce **market-context features** so that the model can incorporate broader market movements rather than relying only on the individual stock's historical behavior.

In [7]:
market_file = RAW_DIR / "SP500_daily.csv"

print("Expected market data file:")
print(market_file.resolve())
print("Exists:", market_file.exists())

Expected market data file:
C:\Users\Aryan\Desktop\AVA-AI_MODEL-2\data\raw\SP500_daily.csv
Exists: False


In [8]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("TWELVE_DATA_API_KEY")

url = "https://api.twelvedata.com/time_series"

params = {
    "symbol": "SPX",
    "interval": "1day",
    "outputsize": 500,
    "apikey": api_key,
}

response = requests.get(url, params=params, timeout=30)
data = response.json()

print("API status:", data.get("status", "success"))
print("Returned rows:", len(data.get("values", [])))
print("Message:", data.get("message", "No error"))

API status: error
Returned rows: 0
Message: This symbol is available starting with the Grow or Venture plan. Consider upgrading now at https://twelvedata.com/pricing


In [9]:
import importlib.util

print("yfinance installed:", importlib.util.find_spec("yfinance") is not None)

yfinance installed: True


In [10]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [11]:
import yfinance as yf

print("yfinance version:", yf.__version__)

yfinance version: 1.7.0


In [12]:
market_data = yf.download(
    "^GSPC",
    period="2y",
    interval="1d",
    auto_adjust=False,
    progress=False,
)

print("Rows:", len(market_data))
print("Columns:", market_data.columns.tolist())
print("Date range:", market_data.index.min(), "to", market_data.index.max())

Rows: 500
Columns: [('Adj Close', '^GSPC'), ('Close', '^GSPC'), ('High', '^GSPC'), ('Low', '^GSPC'), ('Open', '^GSPC'), ('Volume', '^GSPC')]
Date range: 2024-09-30 00:00:00 to 2026-09-28 00:00:00


In [13]:
market_df = market_data.copy()

market_df.columns = [
    "adj_close",
    "close",
    "high",
    "low",
    "open",
    "volume",
]

market_df = market_df.reset_index()
market_df = market_df.rename(columns={"Date": "datetime"})

market_df["datetime"] = pd.to_datetime(market_df["datetime"]).dt.tz_localize(None)

market_df = market_df.sort_values("datetime").reset_index(drop=True)

print(market_df.head())
print("\nRows:", len(market_df))
print("Missing values:", market_df.isna().sum().sum())
print("Date range:", market_df["datetime"].min(), "to", market_df["datetime"].max())

    datetime    adj_close        close         high          low         open  \
0 2024-09-30  5762.479980  5762.479980  5765.140137  5703.529785  5726.520020   
1 2024-10-01  5708.750000  5708.750000  5757.729980  5681.279785  5757.729980   
2 2024-10-02  5709.540039  5709.540039  5719.629883  5674.000000  5698.140137   
3 2024-10-03  5699.939941  5699.939941  5718.779785  5677.370117  5698.189941   
4 2024-10-04  5751.069824  5751.069824  5753.209961  5702.830078  5737.479980   

       volume  
0  4425730000  
1  4025180000  
2  3829050000  
3  3597450000  
4  3479400000  

Rows: 500
Missing values: 0
Date range: 2024-09-30 00:00:00 to 2026-09-28 00:00:00


In [14]:
market_df["market_return"] = market_df["close"].pct_change()
market_df["market_return_lag1"] = market_df["market_return"].shift(1)
market_df["market_return_lag5"] = market_df["market_return"].shift(5)

market_df["market_MA20"] = market_df["close"].rolling(20).mean()
market_df["market_MA50"] = market_df["close"].rolling(50).mean()

market_df["market_vs_MA20"] = (
    market_df["close"] / market_df["market_MA20"] - 1
)

market_df["market_vs_MA50"] = (
    market_df["close"] / market_df["market_MA50"] - 1
)

market_df["market_volatility_20"] = (
    market_df["market_return"].rolling(20).std()
)

market_features = [
    "market_return",
    "market_return_lag1",
    "market_return_lag5",
    "market_vs_MA20",
    "market_vs_MA50",
    "market_volatility_20",
]

print("Market features created:", len(market_features))
print(market_df[["datetime"] + market_features].tail())

Market features created: 6
      datetime  market_return  market_return_lag1  market_return_lag5  \
495 2026-09-22      -0.000008            0.014927           -0.004495   
496 2026-09-23      -0.007548           -0.000008           -0.004472   
497 2026-09-24      -0.000247           -0.007548            0.011381   
498 2026-09-25       0.005099           -0.000247            0.001668   
499 2026-09-28      -0.007712            0.005099            0.014927   

     market_vs_MA20  market_vs_MA50  market_volatility_20  
495        0.012471        0.018179              0.006690  
496        0.004641        0.010063              0.006911  
497        0.004207        0.009465              0.006911  
498        0.009245        0.014055              0.006814  
499        0.001645        0.005639              0.007016  


In [15]:
for ticker in tickers:
    stocks[ticker] = stocks[ticker].merge(
        market_df[["datetime"] + market_features],
        on="datetime",
        how="left",
    )

print("Market context merged.")

for ticker in tickers:
    print(
        f"{ticker}: {len(stocks[ticker])} rows | "
        f"market missing: {stocks[ticker][market_features].isna().any(axis=1).sum()}"
    )

Market context merged.
AAPL: 500 rows | market missing: 50
AMZN: 500 rows | market missing: 50
GOOGL: 500 rows | market missing: 50
MSFT: 500 rows | market missing: 50
NVDA: 500 rows | market missing: 50
TSLA: 500 rows | market missing: 50


In [16]:
v2_market_features = v2_features + market_features

prepared_market_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_market_features + ["target_return"]
    ).copy()

    prepared_market_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_market_features)}"
    )

print("\nTotal features:", len(v2_market_features))

AAPL: 449 usable rows | features=30
AMZN: 449 usable rows | features=30
GOOGL: 449 usable rows | features=30
MSFT: 449 usable rows | features=30
NVDA: 449 usable rows | features=30
TSLA: 449 usable rows | features=30

Total features: 30


In [17]:
market_results = []

for ticker in tickers:
    df = prepared_market_data[ticker]

    X = df[v2_market_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    market_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

market_results_df = pd.DataFrame(market_results)

print(market_results_df.to_string(index=False))

print("\nV2 Market Context overall:")
print("MAE :", market_results_df["MAE"].mean())
print("RMSE:", market_results_df["RMSE"].mean())

C:\Users\Aryan\Desktop\AVA-AI_MODEL-2\.venv\Lib\site-packages\sklearn\utils\parallel.py:161: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001ECC87B6C50>
  for k, v in zip(warning_filter_keys, filter_args)


ticker      MAE     RMSE
  AAPL 0.012721 0.017134
  AMZN 0.017074 0.023561
 GOOGL 0.018118 0.023945
  MSFT 0.014176 0.019432
  NVDA 0.018600 0.023729
  TSLA 0.025093 0.031882

V2 Market Context overall:
MAE : 0.017630259040189365
RMSE: 0.023280504920948047


## Experiment 2 — Market Context Features

V2 was extended with six S&P 500 market-context features:

- Market daily return
- Previous market return
- Five-day lagged market return
- S&P 500 price relative to its 20-day moving average
- S&P 500 price relative to its 50-day moving average
- 20-day market volatility

The market data was obtained independently and merged with each stock dataset by trading date.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012721 | 0.017134 |
| AMZN | 0.017074 | 0.023561 |
| GOOGL | 0.018118 | 0.023945 |
| MSFT | 0.014176 | 0.019432 |
| NVDA | 0.018600 | 0.023729 |
| TSLA | 0.025093 | 0.031882 |

Overall:

- MAE: **0.017630**
- RMSE: **0.023281**

### Interpretation

Adding market-context information improved the expanded V2 technical-feature model.

However, the resulting overall error remained slightly higher than the original V1 Random Forest baseline.

The experiment therefore provides evidence that broad-market information contains useful contextual information for some stocks, but it does not yet establish a consistently superior predictive model.

Further V2 experiments will test additional sources of information while maintaining the same walk-forward validation framework.

In [18]:
market_df["volatility_regime"] = pd.qcut(
    market_df["market_volatility_20"],
    q=3,
    labels=["low", "medium", "high"],
)

print("Volatility regime counts:")
print(market_df["volatility_regime"].value_counts().sort_index())

Volatility regime counts:
volatility_regime
low       160
medium    160
high      160
Name: count, dtype: int64


In [19]:
market_df["volatility_regime_code"] = (
    market_df["volatility_regime"]
    .map({
        "low": 0,
        "medium": 1,
        "high": 2,
    })
)

print(
    market_df[
        ["datetime", "market_volatility_20", "volatility_regime", "volatility_regime_code"]
    ].tail(10)
)

print("\nMissing regime codes:", market_df["volatility_regime_code"].isna().sum())

      datetime  market_volatility_20 volatility_regime volatility_regime_code
490 2026-09-15              0.005709               low                      0
491 2026-09-16              0.005603               low                      0
492 2026-09-17              0.006215               low                      0
493 2026-09-18              0.005918               low                      0
494 2026-09-21              0.006736               low                      0
495 2026-09-22              0.006690               low                      0
496 2026-09-23              0.006911            medium                      1
497 2026-09-24              0.006911            medium                      1
498 2026-09-25              0.006814               low                      0
499 2026-09-28              0.007016            medium                      1

Missing regime codes: 20


In [20]:
regime_features = [
    "volatility_regime_code",
]

for ticker in tickers:
    stocks[ticker] = stocks[ticker].merge(
        market_df[["datetime"] + regime_features],
        on="datetime",
        how="left",
    )

print("Volatility regime merged.")

for ticker in tickers:
    print(
        f"{ticker}: {len(stocks[ticker])} rows | "
        f"missing regime: {stocks[ticker]['volatility_regime_code'].isna().sum()}"
    )

Volatility regime merged.
AAPL: 500 rows | missing regime: 21
AMZN: 500 rows | missing regime: 21
GOOGL: 500 rows | missing regime: 21
MSFT: 500 rows | missing regime: 21
NVDA: 500 rows | missing regime: 21
TSLA: 500 rows | missing regime: 21


In [22]:
v2_regime_features = v2_market_features + [
    "volatility_regime_code"
]

prepared_regime_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_regime_features + ["target_return"]
    ).copy()

    prepared_regime_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features)}"
    )

print("\nTotal features:", len(v2_regime_features))

AAPL: 449 usable rows | features=31
AMZN: 449 usable rows | features=31
GOOGL: 449 usable rows | features=31
MSFT: 449 usable rows | features=31
NVDA: 449 usable rows | features=31
TSLA: 449 usable rows | features=31

Total features: 31


In [23]:
v2_regime_features = v2_market_features + [
    "volatility_regime_code"
]

prepared_regime_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_regime_features + ["target_return"]
    ).copy()

    prepared_regime_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features)}"
    )

print("\nTotal features:", len(v2_regime_features))

AAPL: 449 usable rows | features=31
AMZN: 449 usable rows | features=31
GOOGL: 449 usable rows | features=31
MSFT: 449 usable rows | features=31
NVDA: 449 usable rows | features=31
TSLA: 449 usable rows | features=31

Total features: 31


In [25]:
regime_results = []

for ticker in tickers:
    df = prepared_regime_data[ticker]

    X = df[v2_regime_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )
        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    regime_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

regime_results_df = pd.DataFrame(regime_results)

print(regime_results_df.to_string(index=False))

print("\nV2 Volatility Regime overall:")
print("MAE :", regime_results_df["MAE"].mean())
print("RMSE:", regime_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.012693 0.017104
  AMZN 0.016947 0.023421
 GOOGL 0.018085 0.023898
  MSFT 0.014362 0.019711
  NVDA 0.018659 0.023790
  TSLA 0.025042 0.031787

V2 Volatility Regime overall:
MAE : 0.017631307045384667
RMSE: 0.023285281776429703


In [26]:
regime_results = []

for ticker in tickers:
    df = prepared_regime_data[ticker]

    X = df[v2_regime_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )
        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    regime_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

regime_results_df = pd.DataFrame(regime_results)

print(regime_results_df.to_string(index=False))

print("\nV2 Volatility Regime overall:")
print("MAE :", regime_results_df["MAE"].mean())
print("RMSE:", regime_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.012693 0.017104
  AMZN 0.016947 0.023421
 GOOGL 0.018085 0.023898
  MSFT 0.014362 0.019711
  NVDA 0.018659 0.023790
  TSLA 0.025042 0.031787

V2 Volatility Regime overall:
MAE : 0.017631307045384667
RMSE: 0.023285281776429707


## Experiment 3 — Volatility Regime Feature

V2 was extended with a market volatility-regime feature derived from the S&P 500's 20-day rolling volatility.

The market was divided into three equal-sized volatility regimes:

- Low volatility
- Medium volatility
- High volatility

The regime was encoded numerically and added as an additional model feature.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012693 | 0.017104 |
| AMZN | 0.016947 | 0.023421 |
| GOOGL | 0.018085 | 0.023898 |
| MSFT | 0.014362 | 0.019711 |
| NVDA | 0.018659 | 0.023790 |
| TSLA | 0.025042 | 0.031787 |

Overall:

- MAE: **0.017631**
- RMSE: **0.023285**

### Interpretation

Adding the volatility-regime feature produced almost no change compared with the V2 market-context model.

The market-context model produced:

- MAE: **0.017630**
- RMSE: **0.023281**

The volatility-regime model produced:

- MAE: **0.017631**
- RMSE: **0.023285**

The difference is negligible.

This experiment therefore does not provide evidence that a simple low/medium/high market-volatility regime improves next-day return prediction.

The result is still useful because it narrows the set of V2 features that appear promising.

The next V2 experiment will test **cross-stock relationships**, allowing each stock's model to incorporate recent returns from the other stocks in the universe.

In [27]:
cross_stock_data = {}

for ticker in tickers:
    df = stocks[ticker][["datetime"]].copy()

    for other_ticker in tickers:
        if other_ticker == ticker:
            continue

        other_df = stocks[other_ticker][
            ["datetime", "return"]
        ].rename(
            columns={"return": f"{other_ticker}_return"}
        )

        df = df.merge(
            other_df,
            on="datetime",
            how="left",
        )

    cross_stock_data[ticker] = df

for ticker in tickers:
    print(
        f"{ticker}: {len(cross_stock_data[ticker])} rows | "
        f"cross-stock features={len(tickers) - 1}"
    )

AAPL: 500 rows | cross-stock features=5
AMZN: 500 rows | cross-stock features=5
GOOGL: 500 rows | cross-stock features=5
MSFT: 500 rows | cross-stock features=5
NVDA: 500 rows | cross-stock features=5
TSLA: 500 rows | cross-stock features=5


In [28]:
cross_stock_features = [
    f"{other_ticker}_return"
    for other_ticker in tickers
]

prepared_cross_data = {}

for ticker in tickers:
    df = prepared_regime_data[ticker].merge(
        cross_stock_data[ticker],
        on="datetime",
        how="left",
    )

    all_cross_features = [
        feature for feature in cross_stock_features
        if feature in df.columns
    ]

    df = df.dropna(
        subset=v2_regime_features + all_cross_features + ["target_return"]
    ).copy()

    prepared_cross_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features) + len(all_cross_features)}"
    )

print("\nCross-stock features:", len(cross_stock_features))
print("Total model features:", len(v2_regime_features) + len(cross_stock_features))

AAPL: 449 usable rows | features=36
AMZN: 449 usable rows | features=36
GOOGL: 449 usable rows | features=36
MSFT: 449 usable rows | features=36
NVDA: 449 usable rows | features=36
TSLA: 449 usable rows | features=36

Cross-stock features: 6
Total model features: 37


In [29]:
cross_stock_features_by_ticker = {}

for ticker in tickers:
    cross_stock_features_by_ticker[ticker] = [
        f"{other_ticker}_return"
        for other_ticker in tickers
        if other_ticker != ticker
    ]

for ticker in tickers:
    print(
        f"{ticker}: {len(cross_stock_features_by_ticker[ticker])} "
        f"cross-stock features | "
        f"total features={len(v2_regime_features) + len(cross_stock_features_by_ticker[ticker])}"
    )

AAPL: 5 cross-stock features | total features=36
AMZN: 5 cross-stock features | total features=36
GOOGL: 5 cross-stock features | total features=36
MSFT: 5 cross-stock features | total features=36
NVDA: 5 cross-stock features | total features=36
TSLA: 5 cross-stock features | total features=36


In [30]:
cross_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker]

    cross_features = cross_stock_features_by_ticker[ticker]
    feature_columns = v2_regime_features + cross_features

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )
        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    cross_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

cross_results_df = pd.DataFrame(cross_results)

print(cross_results_df.to_string(index=False))

print("\nV2 Cross-Stock overall:")
print("MAE :", cross_results_df["MAE"].mean())
print("RMSE:", cross_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.012452 0.016916
  AMZN 0.016807 0.023276
 GOOGL 0.017750 0.023687
  MSFT 0.014137 0.019332
  NVDA 0.018818 0.023897
  TSLA 0.024590 0.031230

V2 Cross-Stock overall:
MAE : 0.017425845300088783
RMSE: 0.023056146713284206


## Experiment 4 — Cross-Stock Relationships

V2 was extended with cross-stock return features.

For each stock, the model received the same-day returns of the other five stocks in the six-stock universe.

This allows the model to test whether relationships between stocks provide additional information for next-day return prediction.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012452 | 0.016916 |
| AMZN | 0.016807 | 0.023276 |
| GOOGL | 0.017750 | 0.023687 |
| MSFT | 0.014137 | 0.019332 |
| NVDA | 0.018818 | 0.023897 |
| TSLA | 0.024590 | 0.031230 |

Overall:

- MAE: **0.017426**
- RMSE: **0.023056**

### Interpretation

Adding cross-stock return information improved the V2 volatility-regime model.

The previous V2 volatility-regime model produced:

- MAE: **0.017631**
- RMSE: **0.023285**

The cross-stock model produced:

- MAE: **0.017426**
- RMSE: **0.023056**

The improvement suggests that relationships between stocks may contain useful predictive information.

However, the cross-stock model still did not outperform the original V1 Random Forest baseline:

- V1 MAE: **0.017063**
- V1 RMSE: **0.022595**

Therefore, the result is encouraging as an experimental direction but does not yet justify replacing the Base Model.

The next experiment will investigate whether **different prediction targets** can produce a more stable and meaningful forecasting problem.

In [31]:
direction_data = {}

for ticker in tickers:
    df = prepared_cross_data[ticker].copy()

    df["target_direction"] = (
        df["target_return"] > 0
    ).astype(int)

    direction_data[ticker] = df

    print(
        f"{ticker}: {len(df)} rows | "
        f"positive={df['target_direction'].sum()} | "
        f"negative_or_zero={(df['target_direction'] == 0).sum()}"
    )

AAPL: 449 rows | positive=239 | negative_or_zero=210
AMZN: 449 rows | positive=227 | negative_or_zero=222
GOOGL: 449 rows | positive=231 | negative_or_zero=218
MSFT: 449 rows | positive=230 | negative_or_zero=219
NVDA: 449 rows | positive=234 | negative_or_zero=215
TSLA: 449 rows | positive=226 | negative_or_zero=223


In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

direction_results = []

for ticker in tickers:
    df = direction_data[ticker]

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_direction"]

    fold_accuracy = []
    fold_balanced_accuracy = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_accuracy.append(
            accuracy_score(y_test, predictions)
        )

        fold_balanced_accuracy.append(
            balanced_accuracy_score(y_test, predictions)
        )

    direction_results.append({
        "ticker": ticker,
        "Accuracy": np.mean(fold_accuracy),
        "Balanced_Accuracy": np.mean(fold_balanced_accuracy),
    })

direction_results_df = pd.DataFrame(direction_results)

print(direction_results_df.to_string(index=False))

print("\nV2 Cross-Stock Direction overall:")
print("Accuracy :", direction_results_df["Accuracy"].mean())
print(
    "Balanced Accuracy :",
    direction_results_df["Balanced_Accuracy"].mean()
)

ticker  Accuracy  Balanced_Accuracy
  AAPL  0.458305           0.467480
  AMZN  0.488475           0.494834
 GOOGL  0.488475           0.519833
  MSFT  0.514915           0.538484
  NVDA  0.544915           0.547111
  TSLA  0.468531           0.476583

V2 Cross-Stock Direction overall:
Accuracy : 0.49393596986817334
Balanced Accuracy : 0.507387491762301


In [33]:
direction_baseline_results = []

for ticker in tickers:
    df = direction_data[ticker]
    y = df["target_direction"]

    fold_baseline_accuracy = []

    for train_end, test_start, test_end in walk_splits:
        y_train = y.iloc[:train_end]
        y_test = y.iloc[test_start:test_end]

        majority_class = y_train.mode()[0]
        baseline_predictions = np.full(
            len(y_test),
            majority_class,
        )

        fold_baseline_accuracy.append(
            accuracy_score(y_test, baseline_predictions)
        )

    direction_baseline_results.append({
        "ticker": ticker,
        "Baseline_Accuracy": np.mean(fold_baseline_accuracy),
    })

direction_baseline_df = pd.DataFrame(direction_baseline_results)

print(direction_baseline_df.to_string(index=False))

print("\nOverall majority baseline:")
print(
    direction_baseline_df["Baseline_Accuracy"].mean()
)

ticker  Baseline_Accuracy
  AAPL           0.531864
  AMZN           0.491299
 GOOGL           0.518305
  MSFT           0.505254
  NVDA           0.505028
  TSLA           0.471582

Overall majority baseline:
0.5038888888888888


## Experiment 5 — Cross-Stock Direction Classification

The next-day return target was converted into a binary direction target:

- **1:** next-day return is positive
- **0:** next-day return is zero or negative

A Random Forest classifier was trained using the V2 cross-stock feature set and evaluated with the same walk-forward validation framework.

### Classifier Results

| Ticker | Accuracy | Balanced Accuracy |
|---|---:|---:|
| AAPL | 0.458305 | 0.467480 |
| AMZN | 0.488475 | 0.494834 |
| GOOGL | 0.488475 | 0.519833 |
| MSFT | 0.514915 | 0.538484 |
| NVDA | 0.544915 | 0.547111 |
| TSLA | 0.468531 | 0.476583 |

Overall:

- Accuracy: **49.39%**
- Balanced Accuracy: **50.74%**

### Majority Baseline

A majority-class baseline was calculated independently within each walk-forward training fold.

| Ticker | Majority Baseline Accuracy |
|---|---:|
| AAPL | 0.531864 |
| AMZN | 0.491299 |
| GOOGL | 0.518305 |
| MSFT | 0.505254 |
| NVDA | 0.505028 |
| TSLA | 0.471582 |

Overall majority baseline accuracy:

**50.39%**

### Interpretation

The cross-stock Random Forest classifier achieved **49.39% overall accuracy**, compared with **50.39% for the majority baseline**.

Balanced accuracy was **50.74%**, remaining close to chance-level performance.

Although some individual stocks produced results above their respective majority baselines, the overall experiment did not demonstrate a consistent predictive advantage.

The experiment therefore does not provide evidence that the current V2 feature set can reliably predict next-day stock direction.

This reinforces an important finding from the V1 research: simply changing the model from regression to binary direction classification does not create meaningful predictive signal.

The next V2 experiment will investigate whether a **different return target horizon** provides a more stable forecasting problem.

In [34]:
# Experiment 6 — 5-day forward return target

horizon = 5

target_5d_data = {}

for ticker in tickers:
    df = prepared_cross_data[ticker].copy()

    df["target_return_5d"] = (
        df["close"].shift(-horizon) / df["close"] - 1
    )

    target_5d_data[ticker] = df

    print(
        ticker,
        "| rows:", len(df),
        "| usable 5D targets:",
        df["target_return_5d"].notna().sum()
    )

AAPL | rows: 449 | usable 5D targets: 444
AMZN | rows: 449 | usable 5D targets: 444
GOOGL | rows: 449 | usable 5D targets: 444
MSFT | rows: 449 | usable 5D targets: 444
NVDA | rows: 449 | usable 5D targets: 444
TSLA | rows: 449 | usable 5D targets: 444


In [35]:
target_5d_results = []

for ticker in tickers:
    df = target_5d_data[ticker].dropna(
        subset=["target_return_5d"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return_5d"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    target_5d_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

target_5d_results_df = pd.DataFrame(target_5d_results)

print(target_5d_results_df.to_string(index=False))

print("\nOverall 5-day target:")
print("MAE :", target_5d_results_df["MAE"].mean())
print("RMSE:", target_5d_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.037563 0.044491
  AMZN 0.051177 0.063632
 GOOGL 0.042414 0.055007
  MSFT 0.033061 0.045190
  NVDA 0.040321 0.050708
  TSLA 0.065610 0.083006

Overall 5-day target:
MAE : 0.045024186541636035
RMSE: 0.057005559806887575


In [36]:
target_5d_direction_results = []

for ticker in tickers:
    df = target_5d_data[ticker].dropna(
        subset=["target_return_5d"]
    ).copy()

    df["target_direction_5d"] = (
        df["target_return_5d"] > 0
    ).astype(int)

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_direction_5d"]

    fold_accuracy = []
    fold_balanced_accuracy = []
    fold_baseline_accuracy = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_accuracy.append(
            accuracy_score(y_test, predictions)
        )

        fold_balanced_accuracy.append(
            balanced_accuracy_score(y_test, predictions)
        )

        majority_class = y_train.mode()[0]

        baseline_predictions = np.full(
            len(y_test),
            majority_class,
        )

        fold_baseline_accuracy.append(
            accuracy_score(y_test, baseline_predictions)
        )

    target_5d_direction_results.append({
        "ticker": ticker,
        "Accuracy": np.mean(fold_accuracy),
        "Balanced_Accuracy": np.mean(fold_balanced_accuracy),
        "Baseline_Accuracy": np.mean(fold_baseline_accuracy),
    })

target_5d_direction_results_df = pd.DataFrame(
    target_5d_direction_results
)

print(
    target_5d_direction_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "Accuracy:",
    target_5d_direction_results_df["Accuracy"].mean()
)
print(
    "Balanced Accuracy:",
    target_5d_direction_results_df["Balanced_Accuracy"].mean()
)
print(
    "Majority Baseline:",
    target_5d_direction_results_df["Baseline_Accuracy"].mean()
)

ticker  Accuracy  Balanced_Accuracy  Baseline_Accuracy
  AAPL  0.521111           0.550092           0.463333
  AMZN  0.512963           0.529335           0.514815
 GOOGL  0.452593           0.535755           0.567407
  MSFT  0.525926           0.542100           0.508519
  NVDA  0.550741           0.569935           0.552593
  TSLA  0.422963           0.514270           0.487407

Overall:
Accuracy: 0.497716049382716
Balanced Accuracy: 0.5402479669003245
Majority Baseline: 0.515679012345679


## Experiment 6 — 5-Day Forward Return Target

A second target formulation was tested using the 5-trading-day forward return:

`close[t+5] / close[t] - 1`

Each stock provided 444 usable observations after removing rows without a known 5-day future price.

### 5-Day Regression Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.037563 | 0.044491 |
| AMZN | 0.051177 | 0.063632 |
| GOOGL | 0.042414 | 0.055007 |
| MSFT | 0.033061 | 0.045190 |
| NVDA | 0.040321 | 0.050708 |
| TSLA | 0.065610 | 0.083006 |

Overall:

- MAE: **0.045024**
- RMSE: **0.057006**

The raw error magnitude is larger than the next-day target because the 5-day return naturally has a larger scale. Therefore, direct MAE comparison across different horizons should not be interpreted as a standalone measure of model quality.

### 5-Day Direction Classification

The 5-day return was also converted into a binary direction target.

| Ticker | Accuracy | Balanced Accuracy | Majority Baseline |
|---|---:|---:|---:|
| AAPL | 0.521111 | 0.550092 | 0.463333 |
| AMZN | 0.512963 | 0.529335 | 0.514815 |
| GOOGL | 0.452593 | 0.535755 | 0.567407 |
| MSFT | 0.525926 | 0.542100 | 0.508519 |
| NVDA | 0.550741 | 0.569935 | 0.552593 |
| TSLA | 0.422963 | 0.514270 | 0.487407 |

Overall:

- Accuracy: **49.77%**
- Balanced Accuracy: **54.02%**
- Majority baseline: **51.57%**

### Interpretation

The 5-day direction classifier did not outperform the overall majority baseline in accuracy.

Balanced accuracy was somewhat higher than 50%, indicating some ability to distinguish the two classes after accounting for class balance. However, the improvement was not consistent across stocks, and the dataset is relatively small.

The 5-day target also creates overlapping future-return windows. For example, adjacent observations can share several of the same future trading days. Therefore, this experiment should be treated as exploratory rather than as an independent, fully purged validation of a 5-day trading signal.

Overall, changing the target horizon from one day to five days did not provide clear evidence of a robust predictive advantage.

The next experiments should therefore focus on improving the information available to the model rather than simply changing the forecast horizon.

In [37]:
# Experiment 7 — Feature importance analysis

ticker = "AAPL"

df = prepared_cross_data[ticker].dropna(
    subset=["target_return"]
).copy()

feature_columns = (
    v2_regime_features
    + cross_stock_features_by_ticker[ticker]
)

X = df[feature_columns]
y = df["target_return"]

feature_importance_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

feature_importance_model.fit(X, y)

feature_importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": feature_importance_model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False,
)

print(feature_importance_df.to_string(index=False))

               feature  importance
           MSFT_return    0.067146
        market_vs_MA50    0.046858
     rolling_return_20    0.044877
         volume_change    0.042138
          volatility_5    0.039880
        market_vs_MA20    0.039541
          price_vs_MA5    0.038303
           return_lag3    0.035558
    market_return_lag1    0.033524
     rolling_return_10    0.032843
    market_return_lag5    0.032123
                   MA5    0.030543
      rolling_return_5    0.030083
                 close    0.028263
          return_lag10    0.027815
        volume_vs_MA20    0.027744
           TSLA_return    0.026389
                  MACD    0.026345
         price_vs_MA20    0.025982
  market_volatility_20    0.025517
           return_lag1    0.025430
           NVDA_return    0.025347
          GOOGL_return    0.023018
        MACD_histogram    0.022991
                RSI_14    0.022015
           AMZN_return    0.021604
           return_lag5    0.021031
           return_la

In [38]:
# Experiment 7 — Regime feature ablation

regime_ablation = []

for ticker in tickers:
    regime_mae = v2_regime_results_df.loc[
        v2_regime_results_df["ticker"] == ticker,
        "MAE"
    ].iloc[0]

    market_mae = v2_market_results_df.loc[
        v2_market_results_df["ticker"] == ticker,
        "MAE"
    ].iloc[0]

    regime_rmse = v2_regime_results_df.loc[
        v2_regime_results_df["ticker"] == ticker,
        "RMSE"
    ].iloc[0]

    market_rmse = v2_market_results_df.loc[
        v2_market_results_df["ticker"] == ticker,
        "RMSE"
    ].iloc[0]

    regime_ablation.append({
        "ticker": ticker,
        "Market_MAE": market_mae,
        "Regime_MAE": regime_mae,
        "MAE_Difference": regime_mae - market_mae,
        "Market_RMSE": market_rmse,
        "Regime_RMSE": regime_rmse,
        "RMSE_Difference": regime_rmse - market_rmse,
    })

regime_ablation_df = pd.DataFrame(regime_ablation)

print(regime_ablation_df.to_string(index=False))

print("\nOverall mean differences:")
print(
    "MAE difference:",
    regime_ablation_df["MAE_Difference"].mean()
)
print(
    "RMSE difference:",
    regime_ablation_df["RMSE_Difference"].mean()
)

NameError: name 'v2_regime_results_df' is not defined

In [39]:
[name for name in globals() if "result" in name.lower()]

['v2_results',
 'v2_results_df',
 'market_results',
 'market_results_df',
 'regime_results',
 'regime_results_df',
 'cross_results',
 'cross_results_df',
 'direction_results',
 'direction_results_df',
 'direction_baseline_results',
 'target_5d_results',
 'target_5d_results_df',
 'target_5d_direction_results',
 'target_5d_direction_results_df']

In [40]:
# Experiment 7 — Regime feature ablation

regime_ablation = []

for ticker in tickers:
    regime_row = regime_results_df[
        regime_results_df["ticker"] == ticker
    ].iloc[0]

    market_row = market_results_df[
        market_results_df["ticker"] == ticker
    ].iloc[0]

    regime_ablation.append({
        "ticker": ticker,
        "Market_MAE": market_row["MAE"],
        "Regime_MAE": regime_row["MAE"],
        "MAE_Difference": regime_row["MAE"] - market_row["MAE"],
        "Market_RMSE": market_row["RMSE"],
        "Regime_RMSE": regime_row["RMSE"],
        "RMSE_Difference": regime_row["RMSE"] - market_row["RMSE"],
    })

regime_ablation_df = pd.DataFrame(regime_ablation)

print(regime_ablation_df.to_string(index=False))

print("\nOverall mean differences:")
print(
    "MAE difference:",
    regime_ablation_df["MAE_Difference"].mean()
)
print(
    "RMSE difference:",
    regime_ablation_df["RMSE_Difference"].mean()
)

ticker  Market_MAE  Regime_MAE  MAE_Difference  Market_RMSE  Regime_RMSE  RMSE_Difference
  AAPL    0.012721    0.012693       -0.000028     0.017134     0.017104        -0.000030
  AMZN    0.017074    0.016947       -0.000128     0.023561     0.023421        -0.000140
 GOOGL    0.018118    0.018085       -0.000033     0.023945     0.023898        -0.000047
  MSFT    0.014176    0.014362        0.000186     0.019432     0.019711         0.000279
  NVDA    0.018600    0.018659        0.000059     0.023729     0.023790         0.000061
  TSLA    0.025093    0.025042       -0.000050     0.031882     0.031787        -0.000094

Overall mean differences:
MAE difference: 1.0480051953051447e-06
RMSE difference: 4.7768554816621e-06


## Experiment 7 — Volatility Regime Feature Ablation

A feature-importance analysis showed that `volatility_regime_code` had very low Random Forest importance for AAPL:

**0.15%**

An ablation test was therefore performed to compare the market-context model with and without the volatility regime feature.

### Ablation Results

| Ticker | Market MAE | Regime MAE | MAE Difference | Market RMSE | Regime RMSE | RMSE Difference |
|---|---:|---:|---:|---:|---:|---:|
| AAPL | 0.012721 | 0.012693 | -0.000028 | 0.017134 | 0.017104 | -0.000030 |
| AMZN | 0.017074 | 0.016947 | -0.000128 | 0.023561 | 0.023421 | -0.000140 |
| GOOGL | 0.018118 | 0.018085 | -0.000033 | 0.023945 | 0.023898 | -0.000047 |
| MSFT | 0.014176 | 0.014362 | +0.000186 | 0.019432 | 0.019711 | +0.000279 |
| NVDA | 0.018600 | 0.018659 | +0.000059 | 0.023729 | 0.023790 | +0.000061 |
| TSLA | 0.025093 | 0.025042 | -0.000050 | 0.031882 | 0.031787 | -0.000094 |

Overall mean differences:

- MAE: **+0.00000105**
- RMSE: **+0.00000478**

### Interpretation

The volatility regime feature produced a negligible change in overall validation error.

Some stocks improved slightly while others worsened slightly, with no consistent direction. This is consistent with the very low feature importance observed for `volatility_regime_code`.

The current low/medium/high regime encoding therefore does not demonstrate meaningful predictive value.

For future model versions, the regime feature should not be considered a core predictive feature unless a stronger regime representation demonstrates improvement under validation.

In [41]:
# Experiment 8 — Continuous volatility context

market_df["market_volatility_5"] = (
    market_df["market_return"].rolling(5).std()
)

market_df["market_volatility_10"] = (
    market_df["market_return"].rolling(10).std()
)

market_df["market_stock_vol_ratio"] = (
    market_df["market_volatility_20"]
)

continuous_vol_features = [
    "market_volatility_5",
    "market_volatility_10",
]

for ticker in tickers:
    stock_vol = stocks[ticker][
        ["datetime", "volatility_20"]
    ].rename(
        columns={"volatility_20": "stock_volatility_20"}
    )

    stocks[ticker] = stocks[ticker].merge(
        market_df[
            [
                "datetime",
                "market_volatility_5",
                "market_volatility_10",
            ]
        ],
        on="datetime",
        how="left",
    )

    stocks[ticker]["stock_market_vol_ratio"] = (
        stocks[ticker]["stock_volatility_20"]
        / stocks[ticker]["market_volatility_20"]
    )

continuous_vol_features.append(
    "stock_market_vol_ratio"
)

print("New volatility features:")
print(continuous_vol_features)

KeyError: 'stock_volatility_20'

In [42]:
for ticker in tickers:
    stocks[ticker]["stock_volatility_20"] = (
        stocks[ticker]["return"].rolling(20).std()
    )

    stocks[ticker]["stock_market_vol_ratio"] = (
        stocks[ticker]["stock_volatility_20"]
        / stocks[ticker]["market_volatility_20"]
    )

print("Continuous volatility features:")
print(continuous_vol_features)

Continuous volatility features:
['market_volatility_5', 'market_volatility_10']


In [43]:
continuous_vol_features = [
    "market_volatility_5",
    "market_volatility_10",
    "stock_market_vol_ratio",
]

print("Continuous volatility features:")
print(continuous_vol_features)

Continuous volatility features:
['market_volatility_5', 'market_volatility_10', 'stock_market_vol_ratio']


In [44]:
# Experiment 8 — Continuous volatility context evaluation

continuous_vol_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].copy()

    # Add the new continuous volatility features
    volatility_columns = [
        "market_volatility_5",
        "market_volatility_10",
        "stock_market_vol_ratio",
    ]

    extra_columns = [
        "datetime",
        *volatility_columns,
    ]

    df = df.merge(
        stocks[ticker][extra_columns],
        on="datetime",
        how="left",
        suffixes=("", "_new"),
    )

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
        + volatility_columns
    )

    df = df.dropna(
        subset=feature_columns + ["target_return"]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    continuous_vol_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

continuous_vol_results_df = pd.DataFrame(
    continuous_vol_results
)

print(continuous_vol_results_df.to_string(index=False))

print("\nOverall continuous volatility model:")
print(
    "MAE:",
    continuous_vol_results_df["MAE"].mean()
)
print(
    "RMSE:",
    continuous_vol_results_df["RMSE"].mean()
)

KeyError: "['market_volatility_5', 'market_volatility_10'] not in index"

In [45]:
# Experiment 8 — Continuous volatility context evaluation

continuous_vol_results = []

volatility_columns = [
    "market_volatility_5",
    "market_volatility_10",
    "stock_market_vol_ratio",
]

for ticker in tickers:
    df = prepared_cross_data[ticker].copy()

    # Add market volatility features
    df = df.merge(
        market_df[
            [
                "datetime",
                "market_volatility_5",
                "market_volatility_10",
            ]
        ],
        on="datetime",
        how="left",
    )

    # Add stock/market volatility ratio
    stock_vol = stocks[ticker][
        ["datetime", "stock_volatility_20"]
    ].copy()

    df = df.merge(
        stock_vol,
        on="datetime",
        how="left",
    )

    df["stock_market_vol_ratio"] = (
        df["stock_volatility_20"]
        / df["market_volatility_20"]
    )

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
        + volatility_columns
    )

    df = df.dropna(
        subset=feature_columns + ["target_return"]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    continuous_vol_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

continuous_vol_results_df = pd.DataFrame(
    continuous_vol_results
)

print(continuous_vol_results_df.to_string(index=False))

print("\nOverall continuous volatility model:")
print(
    "MAE:",
    continuous_vol_results_df["MAE"].mean()
)
print(
    "RMSE:",
    continuous_vol_results_df["RMSE"].mean()
)

ticker      MAE     RMSE
  AAPL 0.012466 0.016933
  AMZN 0.016639 0.023062
 GOOGL 0.017235 0.023304
  MSFT 0.014404 0.019650
  NVDA 0.018737 0.023773
  TSLA 0.025055 0.032483

Overall continuous volatility model:
MAE: 0.017422752198277578
RMSE: 0.02320083571347041


## Experiment 8 — Continuous Volatility Context

The simple categorical volatility regime was replaced with continuous volatility information:

- `market_volatility_5`
- `market_volatility_10`
- `stock_market_vol_ratio`

The goal was to preserve more information about the current volatility environment instead of compressing it into low/medium/high categories.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012466 | 0.016933 |
| AMZN | 0.016639 | 0.023062 |
| GOOGL | 0.017235 | 0.023304 |
| MSFT | 0.014404 | 0.019650 |
| NVDA | 0.018737 | 0.023773 |
| TSLA | 0.025055 | 0.032483 |

Overall:

- MAE: **0.017423**
- RMSE: **0.023201**

### Comparison with Cross-Stock V2 Model

The previous cross-stock model achieved:

- MAE: **0.017426**
- RMSE: **0.023056**

The continuous volatility features produced an almost negligible MAE improvement while slightly increasing RMSE.

### Interpretation

The continuous volatility representation does not provide convincing evidence of a meaningful improvement over the existing cross-stock feature set.

This suggests that volatility context, at least in its current form, is not a major source of additional predictive signal.

The feature should therefore remain experimental rather than being treated as a core improvement.

In [46]:
# Experiment 9 — Volatility-normalized target

vol_norm_data = {}

for ticker in tickers:
    df = prepared_cross_data[ticker].copy()

    # Use the stock's own recent volatility
    df["stock_volatility_20"] = (
        df["return"].rolling(20).std()
    )

    # Normalize next-day return by recent volatility
    df["target_vol_normalized"] = (
        df["target_return"] / df["stock_volatility_20"]
    )

    vol_norm_data[ticker] = df

    print(
        ticker,
        "| usable rows:",
        df["target_vol_normalized"].notna().sum()
    )

AAPL | usable rows: 430
AMZN | usable rows: 430
GOOGL | usable rows: 430
MSFT | usable rows: 430
NVDA | usable rows: 430
TSLA | usable rows: 430


In [47]:
# Experiment 9 — Volatility-normalized target evaluation

vol_norm_results = []

for ticker in tickers:
    df = vol_norm_data[ticker].dropna(
        subset=["target_vol_normalized"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_vol_normalized"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    vol_norm_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

vol_norm_results_df = pd.DataFrame(
    vol_norm_results
)

print(vol_norm_results_df.to_string(index=False))

print("\nOverall volatility-normalized target:")
print(
    "MAE:",
    vol_norm_results_df["MAE"].mean()
)
print(
    "RMSE:",
    vol_norm_results_df["RMSE"].mean()
)

ticker      MAE     RMSE
  AAPL 0.932335 1.261034
  AMZN 0.910964 1.315933
 GOOGL 1.360012 1.737776
  MSFT 0.885440 1.263268
  NVDA 0.825028 1.079970
  TSLA 0.881841 1.140892

Overall volatility-normalized target:
MAE: 0.9659364761756701
RMSE: 1.2998121376590042


## Experiment 9 — Volatility-Normalized Target

This experiment changed the prediction target from raw next-day return to a volatility-normalized return:

`target_return / stock_volatility_20`

The objective was to test whether predicting the next-day return relative to the stock's recent volatility produces a more stable learning target.

Each stock had 430 usable observations after accounting for the 20-day volatility window.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.932335 | 1.261034 |
| AMZN | 0.910964 | 1.315933 |
| GOOGL | 1.360012 | 1.737776 |
| MSFT | 0.885440 | 1.263268 |
| NVDA | 0.825028 | 1.079970 |
| TSLA | 0.881841 | 1.140892 |

Overall:

- MAE: **0.965936**
- RMSE: **1.299812**

### Interpretation

The volatility-normalized target changes the scale of the prediction problem substantially, so its MAE and RMSE should not be directly compared with the raw-return models.

This experiment does not by itself demonstrate a meaningful predictive improvement.

The normalized target remains an experimental formulation rather than a replacement for the raw next-day return target.

A proper comparison would require converting the normalized predictions back into raw-return predictions and evaluating them on the original return scale.

In [48]:
# Experiment 9 — Back-transform normalized predictions to raw returns

vol_norm_raw_results = []

for ticker in tickers:
    df = vol_norm_data[ticker].dropna(
        subset=["target_vol_normalized", "stock_volatility_20"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y_normalized = df["target_vol_normalized"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y_normalized.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        normalized_predictions = model.predict(X_test)

        # Convert predictions back to raw return units
        volatility_test = df["stock_volatility_20"].iloc[
            test_start:test_end
        ]

        raw_predictions = (
            normalized_predictions * volatility_test.to_numpy()
        )

        y_raw = df["target_return"].iloc[
            test_start:test_end
        ]

        fold_mae.append(
            mean_absolute_error(y_raw, raw_predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_raw, raw_predictions) ** 0.5
        )

    vol_norm_raw_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

vol_norm_raw_results_df = pd.DataFrame(
    vol_norm_raw_results
)

print(
    vol_norm_raw_results_df.to_string(index=False)
)

print("\nBack-transformed raw-return performance:")
print(
    "MAE:",
    vol_norm_raw_results_df["MAE"].mean()
)
print(
    "RMSE:",
    vol_norm_raw_results_df["RMSE"].mean()
)

ticker      MAE     RMSE
  AAPL 0.013123 0.017269
  AMZN 0.017519 0.025019
 GOOGL 0.024715 0.030382
  MSFT 0.015020 0.021220
  NVDA 0.018793 0.023911
  TSLA 0.024231 0.030800

Back-transformed raw-return performance:
MAE: 0.018900368357651853
RMSE: 0.024766614493985123


### Back-Transformed Raw-Return Evaluation

To make the comparison fair, normalized predictions were converted back to raw-return units using the stock's 20-day volatility.

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.013123 | 0.017269 |
| AMZN | 0.017519 | 0.025019 |
| GOOGL | 0.024715 | 0.030382 |
| MSFT | 0.015020 | 0.021220 |
| NVDA | 0.018793 | 0.023911 |
| TSLA | 0.024231 | 0.030800 |

Overall back-transformed performance:

- MAE: **0.018900**
- RMSE: **0.024767**

For comparison, the cross-stock V2 model achieved:

- MAE: **0.017426**
- RMSE: **0.023056**

### Final Interpretation

After converting predictions back to the original return scale, the volatility-normalized target performs worse than the existing cross-stock model on both MAE and RMSE.

Therefore, the volatility-normalized target is **not adopted as the primary V2 target**.

The original next-day raw return remains the preferred target for subsequent experiments.

In [49]:
# Experiment 10 — Top-feature selection
# Initial diagnostic using AAPL feature importance

top_10_features = (
    feature_importance_df
    .head(10)["feature"]
    .tolist()
)

print("Top 10 features:")
for i, feature in enumerate(top_10_features, start=1):
    print(f"{i}. {feature}")

Top 10 features:
1. MSFT_return
2. market_vs_MA50
3. rolling_return_20
4. volume_change
5. volatility_5
6. market_vs_MA20
7. price_vs_MA5
8. return_lag3
9. market_return_lag1
10. rolling_return_10


In [50]:
# Experiment 10 — AAPL top-10 feature model

ticker = "AAPL"

df = prepared_cross_data[ticker].dropna(
    subset=["target_return"]
).copy()

X = df[top_10_features]
y = df["target_return"]

fold_mae = []
fold_rmse = []

for train_end, test_start, test_end in walk_splits:
    X_train = X.iloc[:train_end]
    y_train = y.iloc[:train_end]

    X_test = X.iloc[test_start:test_end]
    y_test = y.iloc[test_start:test_end]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    fold_mae.append(
        mean_absolute_error(y_test, predictions)
    )

    fold_rmse.append(
        mean_squared_error(y_test, predictions) ** 0.5
    )

print("AAPL top-10 feature model:")
print("MAE:", np.mean(fold_mae))
print("RMSE:", np.mean(fold_rmse))

print("\nAAPL 36-feature cross-stock model:")
print(
    "MAE:",
    cross_results_df.loc[
        cross_results_df["ticker"] == "AAPL",
        "MAE"
    ].iloc[0]
)
print(
    "RMSE:",
    cross_results_df.loc[
        cross_results_df["ticker"] == "AAPL",
        "RMSE"
    ].iloc[0]
)

AAPL top-10 feature model:
MAE: 0.011791457697916875
RMSE: 0.01606504929168864

AAPL 36-feature cross-stock model:
MAE: 0.012452182035147029
RMSE: 0.0169162836270891


In [51]:
# Experiment 10 — Build ticker-specific feature importance rankings

top_features_by_ticker = {}

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X, y)

    importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": model.feature_importances_,
    }).sort_values(
        "importance",
        ascending=False,
    )

    top_features_by_ticker[ticker] = (
        importance_df.head(10)["feature"].tolist()
    )

    print(f"\n{ticker} top 10:")
    for i, feature in enumerate(
        top_features_by_ticker[ticker],
        start=1,
    ):
        print(f"{i}. {feature}")


AAPL top 10:
1. MSFT_return
2. market_vs_MA50
3. rolling_return_20
4. volume_change
5. volatility_5
6. market_vs_MA20
7. price_vs_MA5
8. return_lag3
9. market_return_lag1
10. rolling_return_10

AMZN top 10:
1. MSFT_return
2. return_lag3
3. volume_vs_MA20
4. volume_change
5. MA50
6. market_vs_MA50
7. market_vs_MA20
8. GOOGL_return
9. market_return_lag1
10. market_volatility_20

GOOGL top 10:
1. MACD
2. return_lag3
3. volatility_5
4. market_return_lag1
5. volume_vs_MA20
6. MACD_signal
7. return_lag2
8. NVDA_return
9. AAPL_return
10. return_lag10

MSFT top 10:
1. MA50
2. return_lag10
3. close
4. volume_vs_MA20
5. return_lag1
6. MA20
7. market_return
8. market_vs_MA50
9. volatility_5
10. volume_change

NVDA top 10:
1. volume_change
2. return_lag1
3. return_lag3
4. volume_vs_MA20
5. return_lag2
6. market_vs_MA20
7. volatility_5
8. rolling_return_10
9. AAPL_return
10. MA5

TSLA top 10:
1. close
2. market_vs_MA50
3. return_lag3
4. return_lag10
5. market_vs_MA20
6. volume_vs_MA20
7. MA5
8. ma

In [52]:
# Experiment 10 — Ticker-specific top-10 feature evaluation

top10_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = top_features_by_ticker[ticker]

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    top10_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

top10_results_df = pd.DataFrame(top10_results)

print(
    top10_results_df.to_string(index=False)
)

print("\nOverall ticker-specific top-10 model:")
print(
    "MAE:",
    top10_results_df["MAE"].mean()
)
print(
    "RMSE:",
    top10_results_df["RMSE"].mean()
)

print("\nExisting 36-feature cross-stock model:")
print(
    "MAE:",
    cross_results_df["MAE"].mean()
)
print(
    "RMSE:",
    cross_results_df["RMSE"].mean()
)

ticker      MAE     RMSE
  AAPL 0.011791 0.016065
  AMZN 0.016715 0.023766
 GOOGL 0.015075 0.021247
  MSFT 0.013261 0.018803
  NVDA 0.017991 0.023169
  TSLA 0.024213 0.031379

Overall ticker-specific top-10 model:
MAE: 0.016507890903860095
RMSE: 0.022404753241926246

Existing 36-feature cross-stock model:
MAE: 0.017425845300088783
RMSE: 0.023056146713284206


In [53]:
# Experiment 10 — Leakage-safe ticker-specific feature selection
#
# Feature importance is calculated ONLY on each training fold.
# The selected top-10 features are then evaluated on the untouched test fold.

top10_leakage_safe_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    all_features = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[all_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []
    selected_features_by_fold = []

    for train_end, test_start, test_end in walk_splits:
        X_train_full = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test_full = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        # Feature selection uses training data ONLY
        selector_model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        selector_model.fit(
            X_train_full,
            y_train,
        )

        importance_df = pd.DataFrame({
            "feature": all_features,
            "importance": selector_model.feature_importances_,
        }).sort_values(
            "importance",
            ascending=False,
        )

        top_features = (
            importance_df.head(10)["feature"].tolist()
        )

        selected_features_by_fold.append(top_features)

        # Train final fold model using only selected features
        X_train = X_train_full[top_features]
        X_test = X_test_full[top_features]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X_train,
            y_train,
        )

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    top10_leakage_safe_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

    print(f"\n{ticker} selected features by fold:")
    for i, features in enumerate(
        selected_features_by_fold,
        start=1,
    ):
        print(f"Fold {i}: {features}")

top10_leakage_safe_results_df = pd.DataFrame(
    top10_leakage_safe_results
)

print("\nLeakage-safe ticker-specific top-10 model:")
print(
    top10_leakage_safe_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    top10_leakage_safe_results_df["MAE"].mean()
)
print(
    "RMSE:",
    top10_leakage_safe_results_df["RMSE"].mean()
)


AAPL selected features by fold:
Fold 1: ['market_vs_MA50', 'rolling_return_5', 'rolling_return_20', 'MA5', 'close', 'volume_vs_MA20', 'return_lag3', 'rolling_return_10', 'MACD', 'price_vs_MA5']
Fold 2: ['market_vs_MA50', 'rolling_return_5', 'rolling_return_20', 'price_vs_MA5', 'rolling_return_10', 'MA5', 'return_lag3', 'close', 'market_vs_MA20', 'volume_change']
Fold 3: ['market_vs_MA50', 'rolling_return_20', 'rolling_return_5', 'price_vs_MA5', 'rolling_return_10', 'market_vs_MA20', 'MA5', 'volatility_5', 'return_lag3', 'close']
Fold 4: ['market_vs_MA50', 'price_vs_MA5', 'rolling_return_20', 'rolling_return_5', 'return_lag3', 'rolling_return_10', 'market_vs_MA20', 'MA5', 'close', 'volume_change']
Fold 5: ['market_vs_MA50', 'rolling_return_20', 'rolling_return_5', 'volume_change', 'market_vs_MA20', 'price_vs_MA5', 'return_lag3', 'volatility_5', 'rolling_return_10', 'MA5']

AMZN selected features by fold:
Fold 1: ['return_lag3', 'MA50', 'rolling_return_10', 'MACD', 'MA5', 'market_vs_MA5

## Experiment 10 — Leakage-Safe Feature Selection

Experiment 10 tested whether reducing the expanded V2 feature set from 36 features to 10 ticker-specific features would improve generalization.

The initial experiment selected features using the complete dataset, producing an apparently improved overall MAE of **0.016508** and RMSE of **0.022405**.

However, that feature-selection procedure allowed information from future test periods to influence feature selection.

To correct this, feature importance was recalculated independently inside each walk-forward training fold. Only the top 10 features from each training fold were then used to predict that fold's untouched test period.

### Leakage-Safe Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012600 | 0.017203 |
| AMZN | 0.017255 | 0.024428 |
| GOOGL | 0.018956 | 0.024886 |
| MSFT | 0.013795 | 0.019164 |
| NVDA | 0.018983 | 0.024419 |
| TSLA | 0.025468 | 0.032718 |

Overall:

- MAE: **0.017843**
- RMSE: **0.023803**

For comparison, the existing 36-feature cross-stock model achieved:

- MAE: **0.017426**
- RMSE: **0.023056**

### Interpretation

The apparent improvement from the initial top-10 experiment disappeared after applying leakage-safe feature selection.

The leakage-safe top-10 model performed worse than the full 36-feature cross-stock model on both MAE and RMSE.

Therefore, **feature selection is not adopted as a V2 improvement**.

The fold-specific feature rankings remain useful for diagnostics and understanding which feature groups repeatedly receive importance, but feature importance should not be interpreted as proof of causal or stable predictive relationships.

### Methodological Lesson

Feature selection is part of model training and must be performed using training data only.

Selecting features using the complete dataset before walk-forward validation can leak information from future periods into the model-development process and produce overly optimistic results.

In [54]:
# Experiment 11 — V2 Gradient Boosting with market + cross-stock features

from sklearn.ensemble import GradientBoostingRegressor

gb_v2_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            random_state=42,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    gb_v2_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

gb_v2_results_df = pd.DataFrame(gb_v2_results)

print("V2 Gradient Boosting results:")
print(
    gb_v2_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    gb_v2_results_df["MAE"].mean()
)
print(
    "RMSE:",
    gb_v2_results_df["RMSE"].mean()
)

V2 Gradient Boosting results:
ticker      MAE     RMSE
  AAPL 0.013111 0.017389
  AMZN 0.018415 0.025442
 GOOGL 0.021136 0.027479
  MSFT 0.014207 0.019639
  NVDA 0.019261 0.024343
  TSLA 0.026630 0.033590

Overall:
MAE: 0.01879336978613274
RMSE: 0.02464689804836694


## Experiment 11 — Gradient Boosting

Experiment 11 tested Gradient Boosting Regression using the full V2 market-context and cross-stock feature set.

The model used 300 boosting estimators, a learning rate of 0.03, and maximum tree depth of 2. Evaluation used the same walk-forward validation procedure as the previous experiments.

### Results

| Ticker |      MAE |     RMSE |
| ------ | -------: | -------: |
| AAPL   | 0.013111 | 0.017389 |
| AMZN   | 0.018415 | 0.025442 |
| GOOGL  | 0.021136 | 0.027479 |
| MSFT   | 0.014207 | 0.019639 |
| NVDA   | 0.019261 | 0.024343 |
| TSLA   | 0.026630 | 0.033590 |

Overall:

* MAE: **0.018793**
* RMSE: **0.024647**

The existing V2 Random Forest cross-stock model achieved:

* MAE: **0.017426**
* RMSE: **0.023056**

### Interpretation

Gradient Boosting performed worse than the Random Forest on both MAE and RMSE.

Therefore, Gradient Boosting is **not adopted as the primary V2 regression model**.

This experiment does demonstrate that changing model families alone does not currently reveal a stronger predictive signal in the available feature set.

The Random Forest remains the stronger candidate among the models tested so far, while the naive baseline remains an important reference because the V1 research showed that machine-learning predictions did not consistently outperform it.

### Conclusion

Gradient Boosting does not provide a validated improvement for the current V2 feature set and is not promoted to the primary model.


In [55]:
# Experiment 12 — Random Forest + Gradient Boosting ensemble

ensemble_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        rf_model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        gb_model = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            random_state=42,
        )

        rf_model.fit(X_train, y_train)
        gb_model.fit(X_train, y_train)

        rf_predictions = rf_model.predict(X_test)
        gb_predictions = gb_model.predict(X_test)

        # Equal-weight ensemble
        predictions = (
            0.5 * rf_predictions
            + 0.5 * gb_predictions
        )

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    ensemble_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

ensemble_results_df = pd.DataFrame(
    ensemble_results
)

print("Random Forest + Gradient Boosting ensemble:")
print(
    ensemble_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    ensemble_results_df["MAE"].mean()
)

print(
    "RMSE:",
    ensemble_results_df["RMSE"].mean()
)

Random Forest + Gradient Boosting ensemble:
ticker      MAE     RMSE
  AAPL 0.012634 0.017003
  AMZN 0.017481 0.024205
 GOOGL 0.019305 0.025378
  MSFT 0.014008 0.019302
  NVDA 0.018686 0.023768
  TSLA 0.025294 0.031913

Overall:
MAE: 0.01790152958842359
RMSE: 0.023594668681810472


## Experiment 12 — Random Forest + Gradient Boosting Ensemble

Experiment 12 tested whether combining the V2 Random Forest and Gradient Boosting models could improve prediction through model diversity.

Both models were trained independently within each walk-forward training fold. Their predictions were combined using an equal 50/50 weighting.

### Results

| Ticker |      MAE |     RMSE |
| ------ | -------: | -------: |
| AAPL   | 0.012634 | 0.017003 |
| AMZN   | 0.017481 | 0.024205 |
| GOOGL  | 0.019305 | 0.025378 |
| MSFT   | 0.014008 | 0.019302 |
| NVDA   | 0.018686 | 0.023768 |
| TSLA   | 0.025294 | 0.031913 |

Overall:

* MAE: **0.017902**
* RMSE: **0.023595**

The V2 Random Forest cross-stock model achieved:

* MAE: **0.017426**
* RMSE: **0.023056**

### Interpretation

The equal-weight Random Forest + Gradient Boosting ensemble performed worse than the Random Forest alone on both MAE and RMSE.

Although model blending can reduce error when different models make sufficiently complementary predictions, that benefit was not observed here.

Therefore, the ensemble is **not adopted as the primary V2 model**.

### Conclusion

Neither Gradient Boosting alone nor a simple 50/50 Random Forest + Gradient Boosting ensemble provides a validated improvement over the V2 Random Forest.

The Random Forest remains the preferred V2 candidate among the tested model configurations, while the naive baseline remains an essential benchmark from the V1 research.


In [59]:
# Experiment 13 — Random Forest prediction uncertainty

uncertainty_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []
    fold_correlation = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        # Individual tree predictions
        tree_predictions = np.array([
            tree.predict(X_test.to_numpy())
            for tree in model.estimators_
        ])

        # Mean prediction
        predictions = tree_predictions.mean(axis=0)

        # Prediction dispersion across trees
        prediction_std = tree_predictions.std(axis=0)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

        # Correlation between uncertainty and absolute prediction error
        absolute_errors = np.abs(
            y_test.to_numpy() - predictions
        )

        if (
            np.std(prediction_std) > 0
            and np.std(absolute_errors) > 0
        ):
            correlation = np.corrcoef(
                prediction_std,
                absolute_errors,
            )[0, 1]
        else:
            correlation = np.nan

        fold_correlation.append(correlation)

    uncertainty_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
        "Uncertainty_Error_Correlation": np.nanmean(
            fold_correlation
        ),
    })

uncertainty_results_df = pd.DataFrame(
    uncertainty_results
)

print("Random Forest prediction uncertainty:")
print(
    uncertainty_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    uncertainty_results_df["MAE"].mean()
)

print(
    "RMSE:",
    uncertainty_results_df["RMSE"].mean()
)

print(
    "Mean uncertainty/error correlation:",
    uncertainty_results_df[
        "Uncertainty_Error_Correlation"
    ].mean()
)

Random Forest prediction uncertainty:
ticker      MAE     RMSE  Uncertainty_Error_Correlation
  AAPL 0.012452 0.016916                       0.088893
  AMZN 0.016807 0.023276                       0.285681
 GOOGL 0.017750 0.023687                       0.178213
  MSFT 0.014137 0.019332                       0.142507
  NVDA 0.018818 0.023897                       0.091946
  TSLA 0.024590 0.031230                       0.087656

Overall:
MAE: 0.017425845300088783
RMSE: 0.023056146713284206
Mean uncertainty/error correlation: 0.14581593894927844


## Experiment 13 — Random Forest Prediction Uncertainty

Experiment 13 tested whether the disagreement between individual Random Forest trees could provide a useful estimate of prediction uncertainty.

For each walk-forward validation fold, predictions from all 300 individual trees were collected. The standard deviation across those tree predictions was used as a simple uncertainty measure.

The experiment then measured the correlation between this prediction uncertainty and the model's absolute prediction error.

### Results

| Ticker |      MAE |     RMSE | Uncertainty–Error Correlation |
| ------ | -------: | -------: | ----------------------------: |
| AAPL   | 0.012452 | 0.016916 |                      0.088893 |
| AMZN   | 0.016807 | 0.023276 |                      0.285681 |
| GOOGL  | 0.017750 | 0.023687 |                      0.178213 |
| MSFT   | 0.014137 | 0.019332 |                      0.142507 |
| NVDA   | 0.018818 | 0.023897 |                      0.091946 |
| TSLA   | 0.024590 | 0.031230 |                      0.087656 |

Overall:

* MAE: **0.017426**
* RMSE: **0.023056**
* Mean uncertainty/error correlation: **0.145816**

### Interpretation

The uncertainty measure showed a positive correlation with absolute prediction error for every stock.

The strongest relationship was observed for AMZN (**0.286**), while the remaining stocks showed weaker correlations between approximately **0.088 and 0.178**.

The overall mean correlation of **0.146** indicates that Random Forest tree disagreement contains some information about prediction error, but the relationship is modest.

The MAE and RMSE are identical to the existing V2 cross-stock Random Forest because this experiment does not change the underlying model. It only measures the dispersion of its individual tree predictions.

Therefore, the tree-dispersion measure should be treated as an **experimental uncertainty indicator**, not as a calibrated confidence probability.

### Conclusion

Experiment 13 provides a useful first uncertainty signal for the V2 Random Forest.

However, the current uncertainty estimate is not strong enough to be presented as a reliable probability of prediction correctness.

Future versions can investigate better uncertainty calibration, prediction intervals, conformal methods, or empirical coverage testing.

The Random Forest remains the primary V2 regression candidate among the tested configurations, while the uncertainty measure can be retained as an experimental diagnostic.


In [60]:
# Experiment 14 — Random Forest hyperparameter tuning

rf_tuning_configs = {
    "baseline_300": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0,
    },
    "deeper_regularized": {
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": 1.0,
    },
    "regularized": {
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 3,
        "max_features": 0.8,
    },
    "more_randomized": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
}

rf_tuning_results = []

for config_name, config in rf_tuning_configs.items():

    for ticker in tickers:

        df = prepared_cross_data[ticker].dropna(
            subset=["target_return"]
        ).copy()

        feature_columns = (
            v2_regime_features
            + cross_stock_features_by_ticker[ticker]
        )

        X = df[feature_columns]
        y = df["target_return"]

        fold_mae = []
        fold_rmse = []

        for train_end, test_start, test_end in walk_splits:

            X_train = X.iloc[:train_end]
            y_train = y.iloc[:train_end]

            X_test = X.iloc[test_start:test_end]
            y_test = y.iloc[test_start:test_end]

            model = RandomForestRegressor(
                **config,
                random_state=42,
                n_jobs=-1,
            )

            model.fit(X_train, y_train)

            predictions = model.predict(X_test)

            fold_mae.append(
                mean_absolute_error(y_test, predictions)
            )

            fold_rmse.append(
                mean_squared_error(
                    y_test,
                    predictions
                ) ** 0.5
            )

        rf_tuning_results.append({
            "config": config_name,
            "ticker": ticker,
            "MAE": np.mean(fold_mae),
            "RMSE": np.mean(fold_rmse),
        })

rf_tuning_results_df = pd.DataFrame(
    rf_tuning_results
)

print("Experiment 14 — Random Forest tuning:")
print(
    rf_tuning_results_df.to_string(index=False)
)

print("\nOverall by configuration:")

rf_tuning_summary = (
    rf_tuning_results_df
    .groupby("config")[["MAE", "RMSE"]]
    .mean()
    .sort_values("MAE")
)

print(
    rf_tuning_summary.to_string()
)

Experiment 14 — Random Forest tuning:
            config ticker      MAE     RMSE
      baseline_300   AAPL 0.012452 0.016916
      baseline_300   AMZN 0.016807 0.023276
      baseline_300  GOOGL 0.017750 0.023687
      baseline_300   MSFT 0.014137 0.019332
      baseline_300   NVDA 0.018818 0.023897
      baseline_300   TSLA 0.024590 0.031230
deeper_regularized   AAPL 0.012322 0.016744
deeper_regularized   AMZN 0.016379 0.022836
deeper_regularized  GOOGL 0.016132 0.022114
deeper_regularized   MSFT 0.013611 0.018753
deeper_regularized   NVDA 0.018453 0.023480
deeper_regularized   TSLA 0.024653 0.031251
       regularized   AAPL 0.011991 0.016509
       regularized   AMZN 0.016263 0.022617
       regularized  GOOGL 0.015103 0.021098
       regularized   MSFT 0.013439 0.018657
       regularized   NVDA 0.018340 0.023242
       regularized   TSLA 0.024117 0.030712
   more_randomized   AAPL 0.011859 0.016253
   more_randomized   AMZN 0.015976 0.022311
   more_randomized  GOOGL 0.015457 0.0

## Experiment 14 — Random Forest Hyperparameter Tuning

Experiment 14 tested whether changing the Random Forest hyperparameters could improve the current V2 cross-stock model.

Four configurations were evaluated using the same 36-feature cross-stock dataset and the same walk-forward validation procedure.

### Results

| Configuration      |          MAE |         RMSE |
| ------------------ | -----------: | -----------: |
| Baseline 300 trees |     0.017426 |     0.023056 |
| Deeper regularized |     0.016925 |     0.022530 |
| Regularized        |     0.016542 |     0.022139 |
| More randomized    | **0.016461** | **0.022016** |

The best-performing configuration was:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

### Per-Ticker Results

| Ticker |      MAE |     RMSE |
| ------ | -------: | -------: |
| AAPL   | 0.011859 | 0.016253 |
| AMZN   | 0.015976 | 0.022311 |
| GOOGL  | 0.015457 | 0.021348 |
| MSFT   | 0.013546 | 0.018790 |
| NVDA   | 0.018291 | 0.023238 |
| TSLA   | 0.023638 | 0.030154 |

### Interpretation

The more randomized configuration improved both overall MAE and RMSE compared with the original V2 Random Forest.

Compared with the baseline:

* MAE decreased from **0.017426** to **0.016461**.
* RMSE decreased from **0.023056** to **0.022016**.

The improvement was observed across all six stocks, which makes the result more meaningful than an improvement isolated to a single ticker.

The main changes were increasing feature randomness through `max_features="sqrt"` and adding mild regularization through `min_samples_leaf=2`.

### Conclusion

The `more_randomized` configuration is adopted as the **current V2 Random Forest candidate**.

However, it is not yet declared the final V2 model. Additional validation is required to determine whether the improvement is robust across different walk-forward configurations and is not dependent on the particular validation windows used so far.

The naive baseline remains an important benchmark, and all future improvements must continue to be evaluated using strict out-of-sample validation.


In [66]:
# Experiment 15 — Walk-Forward Robustness (fast diagnostic)

robustness_splits = [
    (140, 140, 200),
    (180, 180, 240),
    (220, 220, 280),
    (260, 260, 320),
    (300, 300, 360),
]

robustness_results = []

for ticker in tickers:

    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in robustness_splits:

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(
                y_test,
                predictions
            ) ** 0.5
        )

    robustness_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

robustness_results_df = pd.DataFrame(
    robustness_results
)

print("Experiment 15 — Walk-Forward Robustness:")
print(
    robustness_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    robustness_results_df["MAE"].mean()
)

print(
    "RMSE:",
    robustness_results_df["RMSE"].mean()
)

print("\nExperiment 14 benchmark:")
print("MAE: 0.016461")
print("RMSE: 0.022016")

Experiment 15 — Walk-Forward Robustness:
ticker      MAE     RMSE
  AAPL 0.010655 0.014749
  AMZN 0.015263 0.020265
 GOOGL 0.014873 0.021046
  MSFT 0.011357 0.015004
  NVDA 0.017619 0.022368
  TSLA 0.023206 0.028851

Overall:
MAE: 0.01549564759657194
RMSE: 0.02038045192526589

Experiment 14 benchmark:
MAE: 0.016461
RMSE: 0.022016


In [67]:
# Experiment 15 — Walk-Forward Robustness

robustness_splits = [
    (140, 140, 200),
    (180, 180, 240),
    (220, 220, 280),
    (260, 260, 320),
    (300, 300, 360),
]

robustness_results = []

for ticker in tickers:

    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in robustness_splits:

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(
                y_test,
                predictions
            ) ** 0.5
        )

    robustness_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

robustness_results_df = pd.DataFrame(
    robustness_results
)

print("Experiment 15 — Walk-Forward Robustness:")
print(
    robustness_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    "MAE:",
    robustness_results_df["MAE"].mean()
)

print(
    "RMSE:",
    robustness_results_df["RMSE"].mean()
)

print("\nExperiment 14 benchmark:")
print("MAE: 0.016461")
print("RMSE: 0.022016")

Experiment 15 — Walk-Forward Robustness:
ticker      MAE     RMSE
  AAPL 0.010660 0.014776
  AMZN 0.015198 0.020308
 GOOGL 0.014936 0.021004
  MSFT 0.011302 0.014970
  NVDA 0.017588 0.022309
  TSLA 0.023491 0.029122

Overall:
MAE: 0.015529339886506649
RMSE: 0.020415049417031494

Experiment 14 benchmark:
MAE: 0.016461
RMSE: 0.022016


## Experiment 15 — Walk-Forward Robustness

Experiment 15 tested whether the improved Random Forest configuration from Experiment 14 remained effective when evaluated using different walk-forward validation windows.

The model configuration was kept unchanged:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

A different set of five chronological walk-forward splits was used:

* 140 → 200
* 180 → 240
* 220 → 280
* 260 → 320
* 300 → 360

### Results

| Ticker |      MAE |     RMSE |
| ------ | -------: | -------: |
| AAPL   | 0.010660 | 0.014776 |
| AMZN   | 0.015198 | 0.020308 |
| GOOGL  | 0.014936 | 0.021004 |
| MSFT   | 0.011302 | 0.014970 |
| NVDA   | 0.017588 | 0.022309 |
| TSLA   | 0.023491 | 0.029122 |

Overall:

* MAE: **0.015529**
* RMSE: **0.020415**

For comparison, Experiment 14 using the original validation windows produced:

* MAE: **0.016461**
* RMSE: **0.022016**

### Interpretation

The Random Forest configuration identified in Experiment 14 continued to perform strongly under a different set of chronological validation windows.

Compared with the Experiment 14 benchmark:

* MAE decreased from **0.016461** to **0.015529**.
* RMSE decreased from **0.022016** to **0.020415**.

All six stocks produced lower MAE and RMSE values under the alternate validation setup.

This provides evidence that the improvement observed in Experiment 14 is not limited to the original validation windows.

However, the alternate walk-forward windows overlap in time and therefore should not be treated as a completely independent out-of-sample test. The results are best interpreted as a robustness check rather than definitive proof of generalization.

### Conclusion

Experiment 15 strengthens confidence in the `more_randomized` Random Forest configuration as the current V2 candidate.

The model continues to outperform its Experiment 14 benchmark under an alternative chronological validation setup.

Further validation should continue to use strict temporal separation and should avoid interpreting overlapping validation windows as independent evidence.

The current candidate remains:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`


In [68]:
# Experiment 16 — Random Forest Number of Trees Sensitivity

tree_configs = [100, 300, 500]

tree_results = []

for n_trees in tree_configs:
    for ticker in tickers:
        df = prepared_cross_data[ticker].dropna(
            subset=["target_return"]
        ).copy()

        feature_columns = (
            v2_regime_features
            + cross_stock_features_by_ticker[ticker]
        )

        X = df[feature_columns]
        y = df["target_return"]

        fold_mae = []
        fold_rmse = []

        for train_end, test_start, test_end in walk_splits:
            X_train = X.iloc[:train_end]
            y_train = y.iloc[:train_end]

            X_test = X.iloc[test_start:test_end]
            y_test = y.iloc[test_start:test_end]

            model = RandomForestRegressor(
                n_estimators=n_trees,
                max_depth=None,
                min_samples_leaf=2,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            )

            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

            fold_mae.append(
                mean_absolute_error(y_test, predictions)
            )
            fold_rmse.append(
                mean_squared_error(y_test, predictions) ** 0.5
            )

        tree_results.append({
            "n_trees": n_trees,
            "ticker": ticker,
            "MAE": np.mean(fold_mae),
            "RMSE": np.mean(fold_rmse),
        })

tree_results_df = pd.DataFrame(tree_results)

print("Experiment 16 — Number of Trees Sensitivity:")
print(tree_results_df.to_string(index=False))

print("\nOverall by number of trees:")
print(
    tree_results_df
    .groupby("n_trees")[["MAE", "RMSE"]]
    .mean()
    .sort_index()
)

Experiment 16 — Number of Trees Sensitivity:
 n_trees ticker      MAE     RMSE
     100   AAPL 0.011988 0.016380
     100   AMZN 0.016096 0.022441
     100  GOOGL 0.015308 0.021291
     100   MSFT 0.013660 0.018832
     100   NVDA 0.018233 0.023241
     100   TSLA 0.023738 0.030246
     300   AAPL 0.011859 0.016253
     300   AMZN 0.015976 0.022311
     300  GOOGL 0.015457 0.021348
     300   MSFT 0.013546 0.018790
     300   NVDA 0.018291 0.023238
     300   TSLA 0.023638 0.030154
     500   AAPL 0.011851 0.016274
     500   AMZN 0.016055 0.022343
     500  GOOGL 0.015458 0.021339
     500   MSFT 0.013557 0.018813
     500   NVDA 0.018243 0.023232
     500   TSLA 0.023742 0.030284

Overall by number of trees:
              MAE      RMSE
n_trees                    
100      0.016504  0.022072
300      0.016461  0.022016
500      0.016484  0.022048


## Experiment 16 — Random Forest Number of Trees Sensitivity

Experiment 16 tested whether the tuned Random Forest model was sensitive to the number of trees in the ensemble.

The remaining hyperparameters were kept fixed:

* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

Three values of `n_estimators` were evaluated:

* 100 trees
* 300 trees
* 500 trees

The same 36-feature cross-stock dataset and original walk-forward validation splits were used.

### Results

| Number of Trees |          MAE |         RMSE |
| --------------: | -----------: | -----------: |
|             100 |     0.016504 |     0.022072 |
|         **300** | **0.016461** | **0.022016** |
|             500 |     0.016484 |     0.022048 |

### Per-Ticker Results

| Trees | Ticker |      MAE |     RMSE |
| ----: | ------ | -------: | -------: |
|   100 | AAPL   | 0.011988 | 0.016380 |
|   100 | AMZN   | 0.016096 | 0.022441 |
|   100 | GOOGL  | 0.015308 | 0.021291 |
|   100 | MSFT   | 0.013660 | 0.018832 |
|   100 | NVDA   | 0.018233 | 0.023241 |
|   100 | TSLA   | 0.023738 | 0.030246 |
|   300 | AAPL   | 0.011859 | 0.016253 |
|   300 | AMZN   | 0.015976 | 0.022311 |
|   300 | GOOGL  | 0.015457 | 0.021348 |
|   300 | MSFT   | 0.013546 | 0.018790 |
|   300 | NVDA   | 0.018291 | 0.023238 |
|   300 | TSLA   | 0.023638 | 0.030154 |
|   500 | AAPL   | 0.011851 | 0.016274 |
|   500 | AMZN   | 0.016055 | 0.022343 |
|   500 | GOOGL  | 0.015458 | 0.021339 |
|   500 | MSFT   | 0.013557 | 0.018813 |
|   500 | NVDA   | 0.018243 | 0.023232 |
|   500 | TSLA   | 0.023742 | 0.030284 |

### Interpretation

The three configurations produced very similar results.

The 300-tree configuration had the lowest overall MAE and RMSE, although the differences between 100, 300, and 500 trees were very small.

Increasing the number of trees from 300 to 500 therefore did not produce a meaningful performance improvement.

### Conclusion

Experiment 16 supports retaining **300 trees** for the current V2 Random Forest candidate.

The current configuration remains:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

Further increases in tree count are not currently justified by the validation results.


In [69]:
# Experiment 17 — Tuned V2 vs Naive Baseline

tuned_vs_naive_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    model_mae = []
    model_rmse = []

    naive_mae = []
    naive_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        # Tuned V2 Random Forest
        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        model_mae.append(
            mean_absolute_error(y_test, predictions)
        )
        model_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

        # Naive baseline:
        # predict next-day return = current-day return
        naive_predictions = df["return"].iloc[
            test_start:test_end
        ].to_numpy()

        naive_mae.append(
            mean_absolute_error(y_test, naive_predictions)
        )
        naive_rmse.append(
            mean_squared_error(y_test, naive_predictions) ** 0.5
        )

    tuned_mae = np.mean(model_mae)
    tuned_rmse = np.mean(model_rmse)
    baseline_mae = np.mean(naive_mae)
    baseline_rmse = np.mean(naive_rmse)

    tuned_vs_naive_results.append({
        "ticker": ticker,
        "Tuned_V2_MAE": tuned_mae,
        "Naive_MAE": baseline_mae,
        "MAE_Improvement": baseline_mae - tuned_mae,
        "Tuned_V2_RMSE": tuned_rmse,
        "Naive_RMSE": baseline_rmse,
        "RMSE_Improvement": baseline_rmse - tuned_rmse,
    })

tuned_vs_naive_results_df = pd.DataFrame(
    tuned_vs_naive_results
)

print("Experiment 17 — Tuned V2 vs Naive Baseline:")
print(
    tuned_vs_naive_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    tuned_vs_naive_results_df[
        [
            "Tuned_V2_MAE",
            "Naive_MAE",
            "MAE_Improvement",
            "Tuned_V2_RMSE",
            "Naive_RMSE",
            "RMSE_Improvement",
        ]
    ].mean()
)

Experiment 17 — Tuned V2 vs Naive Baseline:
ticker  Tuned_V2_MAE  Naive_MAE  MAE_Improvement  Tuned_V2_RMSE  Naive_RMSE  RMSE_Improvement
  AAPL      0.011859   0.015784         0.003924       0.016253    0.020687          0.004434
  AMZN      0.015976   0.021111         0.005135       0.022311    0.028240          0.005930
 GOOGL      0.015457   0.020679         0.005223       0.021348    0.026811          0.005462
  MSFT      0.013546   0.018403         0.004858       0.018790    0.025100          0.006310
  NVDA      0.018291   0.025691         0.007400       0.023238    0.033393          0.010155
  TSLA      0.023638   0.032497         0.008859       0.030154    0.042307          0.012153

Overall:
Tuned_V2_MAE        0.016461
Naive_MAE           0.022361
MAE_Improvement     0.005900
Tuned_V2_RMSE       0.022016
Naive_RMSE          0.029423
RMSE_Improvement    0.007407
dtype: float64


## Experiment 17 — Tuned V2 vs Naive Baseline

Experiment 17 directly compared the current tuned V2 Random Forest against the naive next-day return baseline.

The tuned V2 model used:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

Both models were evaluated using the same original walk-forward validation splits.

### Results

| Ticker | Tuned V2 MAE | Naive MAE | MAE Improvement | Tuned V2 RMSE | Naive RMSE | RMSE Improvement |
| ------ | -----------: | --------: | --------------: | ------------: | ---------: | ---------------: |
| AAPL   |     0.011859 |  0.015784 |        0.003924 |      0.016253 |   0.020687 |         0.004434 |
| AMZN   |     0.015976 |  0.021111 |        0.005135 |      0.022311 |   0.028240 |         0.005930 |
| GOOGL  |     0.015457 |  0.020679 |        0.005223 |      0.021348 |   0.026811 |         0.005462 |
| MSFT   |     0.013546 |  0.018403 |        0.004858 |      0.018790 |   0.025100 |         0.006310 |
| NVDA   |     0.018291 |  0.025691 |        0.007400 |      0.023238 |   0.033393 |         0.010155 |
| TSLA   |     0.023638 |  0.032497 |        0.008859 |      0.030154 |   0.042307 |         0.012153 |

### Overall Results

| Model                  |          MAE |         RMSE |
| ---------------------- | -----------: | -----------: |
| Tuned V2 Random Forest | **0.016461** | **0.022016** |
| Naive baseline         |     0.022361 |     0.029423 |

The tuned V2 model reduced MAE by **0.005900** and RMSE by **0.007407** relative to the naive baseline.

This corresponds to approximately:

* **26.4% lower MAE**
* **25.2% lower RMSE**

### Interpretation

The tuned V2 Random Forest produced lower MAE and RMSE than the naive baseline for all six evaluated stocks.

This is an important result because previous V1 experiments did not consistently outperform the naive benchmark.

The result indicates that the expanded V2 feature set, cross-stock context, and tuned Random Forest configuration provide useful predictive information under the current walk-forward validation procedure.

However, these results measure prediction error rather than trading profitability. They should not be interpreted as evidence of guaranteed future returns or as a validated trading strategy.

The evaluation also uses the same historical period and validation framework used throughout the V2 experiments. Additional untouched holdout testing remains necessary before treating the model as fully validated.

### Conclusion

Experiment 17 provides the strongest evidence so far that the current V2 model improves upon the naive return-prediction baseline.

The current V2 candidate remains:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

Further validation should focus on whether this improvement remains present on genuinely unseen data and under additional robustness checks.


In [70]:
# Experiment 18 — Untouched Final Holdout

holdout_size = 60

holdout_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    train_end = len(df) - holdout_size

    X_train = X.iloc[:train_end]
    y_train = y.iloc[:train_end]

    X_holdout = X.iloc[train_end:]
    y_holdout = y.iloc[train_end:]

    # Tuned V2 Random Forest
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    v2_predictions = model.predict(X_holdout)

    v2_mae = mean_absolute_error(
        y_holdout,
        v2_predictions
    )
    v2_rmse = mean_squared_error(
        y_holdout,
        v2_predictions
    ) ** 0.5

    # Naive baseline:
    # predict next-day return = current-day return
    naive_predictions = df["return"].iloc[
        train_end:
    ].to_numpy()

    naive_mae = mean_absolute_error(
        y_holdout,
        naive_predictions
    )
    naive_rmse = mean_squared_error(
        y_holdout,
        naive_predictions
    ) ** 0.5

    holdout_results.append({
        "ticker": ticker,
        "Tuned_V2_MAE": v2_mae,
        "Naive_MAE": naive_mae,
        "MAE_Improvement": naive_mae - v2_mae,
        "Tuned_V2_RMSE": v2_rmse,
        "Naive_RMSE": naive_rmse,
        "RMSE_Improvement": naive_rmse - v2_rmse,
    })

holdout_results_df = pd.DataFrame(holdout_results)

print("Experiment 18 — Untouched Final Holdout:")
print(holdout_results_df.to_string(index=False))

print("\nOverall Holdout Results:")
print(
    holdout_results_df[
        [
            "Tuned_V2_MAE",
            "Naive_MAE",
            "MAE_Improvement",
            "Tuned_V2_RMSE",
            "Naive_RMSE",
            "RMSE_Improvement",
        ]
    ].mean()
)

print(f"\nHoldout observations per ticker: {holdout_size}")
print(f"Training observations per ticker: {train_end}")

Experiment 18 — Untouched Final Holdout:
ticker  Tuned_V2_MAE  Naive_MAE  MAE_Improvement  Tuned_V2_RMSE  Naive_RMSE  RMSE_Improvement
  AAPL      0.012710   0.017481         0.004771       0.018057    0.022200          0.004143
  AMZN      0.016188   0.021704         0.005516       0.027629    0.031771          0.004142
 GOOGL      0.015779   0.021847         0.006068       0.021455    0.028606          0.007151
  MSFT      0.016520   0.022571         0.006050       0.026804    0.034341          0.007537
  NVDA      0.019888   0.027573         0.007685       0.025454    0.037490          0.012036
  TSLA      0.024445   0.039997         0.015551       0.034130    0.052751          0.018621

Overall Holdout Results:
Tuned_V2_MAE        0.017588
Naive_MAE           0.025195
MAE_Improvement     0.007607
Tuned_V2_RMSE       0.025588
Naive_RMSE          0.034527
RMSE_Improvement    0.008938
dtype: float64

Holdout observations per ticker: 60
Training observations per ticker: 389


## Experiment 18 — Untouched Final Holdout

Experiment 18 evaluated the current V2 Random Forest on a final chronological holdout period that was not used during the previous model-selection experiments.

The most recent **60 usable observations** for each stock were reserved as the holdout set.

The tuned V2 configuration was kept fixed:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

For each ticker:

* 389 observations were used for training.
* 60 observations were reserved for final holdout evaluation.

The tuned V2 model was compared directly against the naive baseline.

### Results

| Ticker | Tuned V2 MAE | Naive MAE | MAE Improvement | Tuned V2 RMSE | Naive RMSE | RMSE Improvement |
| ------ | -----------: | --------: | --------------: | ------------: | ---------: | ---------------: |
| AAPL   |     0.012710 |  0.017481 |        0.004771 |      0.018057 |   0.022200 |         0.004143 |
| AMZN   |     0.016188 |  0.021704 |        0.005516 |      0.027629 |   0.031771 |         0.004142 |
| GOOGL  |     0.015779 |  0.021847 |        0.006068 |      0.021455 |   0.028606 |         0.007151 |
| MSFT   |     0.016520 |  0.022571 |        0.006050 |      0.026804 |   0.034341 |         0.007537 |
| NVDA   |     0.019888 |  0.027573 |        0.007685 |      0.025454 |   0.037490 |         0.012036 |
| TSLA   |     0.024445 |  0.039997 |        0.015551 |      0.034130 |   0.052751 |         0.018621 |

### Overall Holdout Results

| Model                      |          MAE |         RMSE |
| -------------------------- | -----------: | -----------: |
| **Tuned V2 Random Forest** | **0.017588** | **0.025588** |
| Naive baseline             |     0.025195 |     0.034527 |

The tuned V2 model reduced:

* MAE by **0.007607**
* RMSE by **0.008938**

Relative to the naive baseline, this corresponds to approximately:

* **30.2% lower MAE**
* **25.9% lower RMSE**

### Interpretation

The tuned V2 Random Forest produced lower prediction error than the naive baseline on the final holdout period for all six evaluated stocks.

This result is important because the holdout observations were not used to select the Random Forest hyperparameters or feature configuration.

The result therefore provides stronger evidence of out-of-sample robustness than the earlier experiments that reused the original walk-forward evaluation windows.

The holdout results are also consistent across all six stocks rather than being driven by a single ticker.

However, the evaluation remains limited to the historical dataset and the selected six stocks. It measures prediction error rather than trading profitability and does not establish that the model will continue to outperform the baseline in future market conditions.

### Conclusion

Experiment 18 provides the strongest validation evidence obtained so far for the current V2 model.

The tuned Random Forest continues to outperform the naive baseline on an untouched chronological holdout period across all six stocks.

The current V2 configuration remains:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

The model can now move toward additional validation such as directional performance, prediction-error distribution, and eventually a properly defined trading/backtesting framework.

The final holdout should be treated as a completed evaluation and should not be reused for further model tuning.


In [71]:
# Experiment 19 — Final Holdout Directional Accuracy

direction_holdout_results = []

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    holdout_size = 60
    train_end = len(df) - holdout_size

    X_train = X.iloc[:train_end]
    y_train = y.iloc[:train_end]

    X_holdout = X.iloc[train_end:]
    y_holdout = y.iloc[train_end:]

    # Tuned V2 Random Forest
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_holdout)

    actual_direction = (y_holdout.to_numpy() > 0).astype(int)
    predicted_direction = (predictions > 0).astype(int)

    accuracy = np.mean(
        predicted_direction == actual_direction
    )

    balanced_accuracy = balanced_accuracy_score(
        actual_direction,
        predicted_direction,
    )

    # Majority-direction baseline based only on training data
    train_direction = (y_train.to_numpy() > 0).astype(int)
    majority_class = int(
        np.mean(train_direction) >= 0.5
    )

    baseline_predictions = np.full(
        len(actual_direction),
        majority_class,
    )

    baseline_accuracy = np.mean(
        baseline_predictions == actual_direction
    )

    baseline_balanced_accuracy = balanced_accuracy_score(
        actual_direction,
        baseline_predictions,
    )

    direction_holdout_results.append({
        "ticker": ticker,
        "V2_Accuracy": accuracy,
        "Baseline_Accuracy": baseline_accuracy,
        "Accuracy_Difference": accuracy - baseline_accuracy,
        "V2_Balanced_Accuracy": balanced_accuracy,
        "Baseline_Balanced_Accuracy": baseline_balanced_accuracy,
        "Balanced_Accuracy_Difference": (
            balanced_accuracy - baseline_balanced_accuracy
        ),
    })

direction_holdout_results_df = pd.DataFrame(
    direction_holdout_results
)

print("Experiment 19 — Final Holdout Directional Accuracy:")
print(
    direction_holdout_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    direction_holdout_results_df[
        [
            "V2_Accuracy",
            "Baseline_Accuracy",
            "Accuracy_Difference",
            "V2_Balanced_Accuracy",
            "Baseline_Balanced_Accuracy",
            "Balanced_Accuracy_Difference",
        ]
    ].mean()
)

Experiment 19 — Final Holdout Directional Accuracy:
ticker  V2_Accuracy  Baseline_Accuracy  Accuracy_Difference  V2_Balanced_Accuracy  Baseline_Balanced_Accuracy  Balanced_Accuracy_Difference
  AAPL     0.583333           0.566667             0.016667              0.623303                         0.5                      0.123303
  AMZN     0.583333           0.400000             0.183333              0.486111                         0.5                     -0.013889
 GOOGL     0.566667           0.483333             0.083333              0.568409                         0.5                      0.068409
  MSFT     0.433333           0.583333            -0.150000              0.502857                         0.5                      0.002857
  NVDA     0.616667           0.500000             0.116667              0.616667                         0.5                      0.116667
  TSLA     0.550000           0.466667             0.083333              0.546875                         0.

## Experiment 19 — Final Holdout Directional Accuracy

Experiment 19 evaluated whether the tuned V2 Random Forest could correctly predict the direction of the next-day return on the untouched final holdout period.

The tuned V2 configuration was:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

The final holdout contained 60 observations per ticker.

Directional accuracy was compared against a majority-direction baseline determined from the training data.

### Results

| Ticker | V2 Accuracy | Baseline Accuracy | Accuracy Difference | V2 Balanced Accuracy | Baseline Balanced Accuracy | Balanced Accuracy Difference |
| ------ | ----------: | ----------------: | ------------------: | -------------------: | -------------------------: | ---------------------------: |
| AAPL   |      58.33% |            56.67% |            +1.67 pp |               62.33% |                     50.00% |                    +12.33 pp |
| AMZN   |      58.33% |            40.00% |           +18.33 pp |               48.61% |                     50.00% |                     -1.39 pp |
| GOOGL  |      56.67% |            48.33% |            +8.33 pp |               56.84% |                     50.00% |                     +6.84 pp |
| MSFT   |      43.33% |            58.33% |           -15.00 pp |               50.29% |                     50.00% |                     +0.29 pp |
| NVDA   |      61.67% |            50.00% |           +11.67 pp |               61.67% |                     50.00% |                    +11.67 pp |
| TSLA   |      55.00% |            46.67% |            +8.33 pp |               54.69% |                     50.00% |                     +4.69 pp |

### Overall Results

| Metric            |         V2 | Baseline |   Difference |
| ----------------- | ---------: | -------: | -----------: |
| Accuracy          | **55.56%** |   50.00% | **+5.56 pp** |
| Balanced Accuracy | **55.74%** |   50.00% | **+5.74 pp** |

### Interpretation

The tuned V2 model achieved higher aggregate directional accuracy and balanced accuracy than the majority-direction baseline on the untouched final holdout.

The overall accuracy was **55.56%**, compared with **50.00%** for the baseline.

The overall balanced accuracy was **55.74%**, compared with **50.00%** for the baseline.

The per-ticker results were mixed. V2 produced higher accuracy than the baseline for AAPL, AMZN, GOOGL, NVDA, and TSLA, while MSFT had lower accuracy.

Balanced accuracy also varied by ticker. AAPL, GOOGL, NVDA, and TSLA were above the baseline, while AMZN was slightly below it and MSFT was approximately equal to the baseline.

Therefore, the aggregate directional result is encouraging, but it should not be interpreted as evidence of a universally reliable directional signal across individual stocks.

The holdout contains only 60 observations per ticker, so the directional estimates remain subject to substantial sampling variation.

### Conclusion

Experiment 19 provides additional out-of-sample evidence that the tuned V2 model contains some directional information during the final holdout period.

The aggregate result exceeded the majority-direction baseline on both accuracy and balanced accuracy.

However, performance was not uniform across stocks, and the sample size of 60 holdout observations per ticker is limited.

The final holdout remains locked and should not be reused for further model tuning.

The current V2 candidate remains:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`


In [72]:
# Experiment 20 — Final Holdout Prediction & Error Analysis

error_analysis_results = []
holdout_predictions = {}

for ticker in tickers:
    df = prepared_cross_data[ticker].dropna(
        subset=["target_return"]
    ).copy()

    feature_columns = (
        v2_regime_features
        + cross_stock_features_by_ticker[ticker]
    )

    X = df[feature_columns]
    y = df["target_return"]

    holdout_size = 60
    train_end = len(df) - holdout_size

    X_train = X.iloc[:train_end]
    y_train = y.iloc[:train_end]

    X_holdout = X.iloc[train_end:]
    y_holdout = y.iloc[train_end:]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_holdout)
    actuals = y_holdout.to_numpy()

    errors = actuals - predictions
    absolute_errors = np.abs(errors)

    holdout_predictions[ticker] = pd.DataFrame({
        "actual_return": actuals,
        "predicted_return": predictions,
        "error": errors,
        "absolute_error": absolute_errors,
    })

    error_analysis_results.append({
        "ticker": ticker,
        "MAE": np.mean(absolute_errors),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "Mean_Predicted": np.mean(predictions),
        "Median_Predicted": np.median(predictions),
        "Std_Predicted": np.std(predictions),
        "Mean_Actual": np.mean(actuals),
        "Median_Actual": np.median(actuals),
        "Std_Actual": np.std(actuals),
        "Prediction_Actual_Correlation": np.corrcoef(
            predictions, actuals
        )[0, 1],
        "Mean_Error_Bias": np.mean(errors),
        "Max_Absolute_Error": np.max(absolute_errors),
    })

error_analysis_results_df = pd.DataFrame(
    error_analysis_results
)

print("Experiment 20 — Final Holdout Prediction & Error Analysis:")
print(
    error_analysis_results_df.to_string(index=False)
)

print("\nOverall:")
print(
    error_analysis_results_df[
        [
            "MAE",
            "RMSE",
            "Mean_Predicted",
            "Median_Predicted",
            "Std_Predicted",
            "Mean_Actual",
            "Median_Actual",
            "Std_Actual",
            "Prediction_Actual_Correlation",
            "Mean_Error_Bias",
            "Max_Absolute_Error",
        ]
    ].mean()
)

print("\nLargest absolute prediction errors:")
for ticker in tickers:
    largest_errors = (
        holdout_predictions[ticker]
        .sort_values("absolute_error", ascending=False)
        .head(3)
    )

    print(f"\n{ticker}:")
    print(largest_errors.to_string(index=False))

Experiment 20 — Final Holdout Prediction & Error Analysis:
ticker      MAE     RMSE  Mean_Predicted  Median_Predicted  Std_Predicted  Mean_Actual  Median_Actual  Std_Actual  Prediction_Actual_Correlation  Mean_Error_Bias  Max_Absolute_Error
  AAPL 0.012710 0.018057       -0.001655         -0.002408       0.003631     0.002559       0.002691    0.017634                       0.123651         0.004214            0.067303
  AMZN 0.016188 0.027629       -0.004935         -0.004783       0.002171     0.000864      -0.003478    0.026202                      -0.338312         0.005800            0.164734
 GOOGL 0.015779 0.021455        0.001023          0.000531       0.004775    -0.000564      -0.000526    0.021725                       0.178175        -0.001587            0.068678
  MSFT 0.016520 0.026804       -0.003521         -0.003226       0.003134     0.005235       0.002663    0.025219                       0.025540         0.008755            0.156404
  NVDA 0.019888 0.025454       

## Experiment 20 — Final Holdout Prediction and Error Analysis

Experiment 20 analyzed the predictions produced by the frozen V2 Random Forest on the final 60-observation holdout period.

The purpose was to examine not only MAE and RMSE, but also prediction distribution, prediction bias, prediction-to-actual correlation, and the largest individual errors.

The model configuration remained unchanged:

* `n_estimators=300`
* `max_depth=None`
* `min_samples_leaf=2`
* `max_features="sqrt"`

### Results

| Ticker |      MAE |     RMSE | Mean Predicted | Mean Actual | Prediction–Actual Correlation | Mean Error Bias |
| ------ | -------: | -------: | -------------: | ----------: | ----------------------------: | --------------: |
| AAPL   | 0.012710 | 0.018057 |      -0.001655 |    0.002559 |                      0.123651 |        0.004214 |
| AMZN   | 0.016188 | 0.027629 |      -0.004935 |    0.000864 |                     -0.338312 |        0.005800 |
| GOOGL  | 0.015779 | 0.021455 |       0.001023 |   -0.000564 |                      0.178175 |       -0.001587 |
| MSFT   | 0.016520 | 0.026804 |      -0.003521 |    0.005235 |                      0.025540 |        0.008755 |
| NVDA   | 0.019888 | 0.025454 |      -0.004956 |    0.002379 |                      0.141333 |        0.007334 |
| TSLA   | 0.024445 | 0.034130 |      -0.000174 |   -0.001666 |                     -0.010406 |       -0.001492 |

### Overall Results

| Metric                           |         Value |
| -------------------------------- | ------------: |
| MAE                              |  **0.017588** |
| RMSE                             |  **0.025588** |
| Mean predicted return            | **-0.002370** |
| Median predicted return          | **-0.002601** |
| Prediction standard deviation    |  **0.004159** |
| Mean actual return               |  **0.001468** |
| Median actual return             | **-0.000402** |
| Actual-return standard deviation |  **0.024792** |
| Prediction–actual correlation    |  **0.019997** |
| Mean error bias                  |  **0.003837** |
| Maximum absolute error           |  **0.115285** |

### Interpretation

The analysis shows that the V2 Random Forest predictions are substantially less variable than the actual returns.

The standard deviation of predicted returns was approximately **0.00416**, compared with **0.02479** for actual returns.

This indicates that the model tends to produce predictions close to zero and does not reproduce the full magnitude of large market movements.

The overall prediction-to-actual correlation was only **0.020**, indicating a very weak linear relationship between the predicted return magnitude and the realized return magnitude during the final holdout.

The mean prediction was **-0.237%**, while the mean actual return was **+0.147%**, producing an overall positive error bias of approximately **0.384 percentage points** under the definition `actual - prediction`.

The largest individual errors also occurred during large market movements. Examples included:

* AMZN actual return of approximately **+15.32%** versus prediction of **-1.15%**.
* MSFT actual return of approximately **+15.51%** versus prediction of **-0.13%**.
* TSLA actual return of approximately **-14.52%** versus prediction of **-0.12%**.
* NVDA actual return of approximately **+8.74%** versus prediction of **-0.32%**.

These observations show that the model's relatively good aggregate MAE/RMSE does not mean it can accurately forecast the magnitude of extreme daily moves.

### Relationship to Experiment 19

Experiment 19 showed aggregate directional accuracy of **55.56%** and balanced accuracy of **55.74%** on the same holdout.

Experiment 20 shows why these results should be interpreted separately from regression accuracy.

The model can contain some directional information while still producing highly compressed return estimates that fail to capture the magnitude of large moves.

### Conclusion

Experiment 20 identifies an important limitation of the current V2 regression model.

The model has demonstrated lower prediction error than the naive baseline on the final holdout, but its predictions are strongly compressed toward zero and have very low correlation with actual return magnitudes.

Therefore, the current V2 model should be described as having **useful but limited predictive information**, rather than as an accurate estimator of future return magnitude.

The final holdout remains locked and should not be used for further tuning.

Future development can investigate alternative target formulations, calibrated prediction intervals, regime-aware models, or models specifically designed to handle large-return events.
